# Reproducing Jeong, Taylor, Floeder et al. (2022) from dandiset 000351

**Paper.** Jeong H, Taylor A, Floeder JR, Lohmann M, Mihalas S, Wu B, Zhou M, Burke DA,
Namboodiri VMK (2022). Mesolimbic dopamine release conveys causal associations. *Science* 378,
eabq6740. [doi:10.1126/science.abq6740](https://doi.org/10.1126/science.abq6740). The claims file
(`claims/000351/10.1126-science.abq6740.yaml`) was written from the accepted author manuscript in
PubMed Central (PMC9910357), its figure legends, and its supplement (Materials and Methods,
Supplementary Notes, Table S1). Its criteria cite the Methods and the paper's code,
github.com/namboodirilab/ANCCR at commit cff44144e60a4313a30ffbb4bb8a1f3dd398b525.

**Dandiset.** [DANDI:000351](https://dandiarchive.org/dandiset/000351/0.260828.1905), version
0.260828.1905. The paper cites the dandiset without a version (its code reads the draft). This is
the only published version, so it is pinned. The draft has the same 428 assets and size; it is
not used, since it can change.

**What the dandiset holds.** Photometry (dF/F of dLight1.3b in the nucleus accumbens core) and the
event log (licks, rewards, cues, laser) of the 8 mice of Experiments 1 to 6, the 14 mice of
Experiment 7, and the event logs (no photometry) of the 6 mice of Fig S8L. The simulations of the
paper (Figs 2, 3C and F, 5B, 6H, S5, S6, S13) use no recorded data.

**Outside inputs.** The day lists (startday, endday, learnedday) that the paper's Matlab scripts
use to select sessions are copied into the helper `jeong.py` from the ANCCR repository at the
commit above, and the notebook reads the scripts at that commit to check the copies; they name sessions by the Day number in the dandiset's file names
(claim session-day-link). The spreadsheet that the dandiset's description names ("Subject and
session information", a Google spreadsheet) is not used: it has no fixed version to pin.

| Claim | Tested here |
|---|---|
| session-day-link, methods-session-counts | yes |
| exp1-excluded-rewards, fig3e, fig3g, figs8k, figs8b, figs8d, figs8g, figs8h, figs8l | yes |
| figs8e | tested; blocked, methods_underspecified (the readings differ) |
| fig4ef, figs10c, fig4hi (two), figs10b, fig4kl, fig4jl, fig4p, fig6b | yes |
| fig5d (two), figs11de | yes |
| fig6d (two), fig6g, fig6i (three), figs12f | yes |
| fig2, fig3cf, fig5b, fig6h, figs6, figs5, figs13 (simulations) | blocked, not_runnable |

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import pingouin as pg
import statsmodels.api as sm
from scipy import stats

import jeong

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 400)

started = time.time()
assets = jeong.list_assets()
photometry_assets = [a for a in assets if not a["subject"].startswith("DB-")]
lick_only_assets = [a for a in assets if a["subject"].startswith("DB-")]
sessions = jeong.load_many(photometry_assets, dff=True)
sessions += jeong.load_many(lick_only_assets, dff=False)
print(f"{len(assets)} assets in dandiset {jeong.DANDISET_ID} version {jeong.DANDISET_VERSION}; "
      f"loaded {len(sessions)} sessions in {time.time() - started:.0f} s")


def subject_sessions(subject, task=None):
    return sorted([s for s in sessions if s["subject"] == subject
                   and (task is None or s["description"] == task)], key=lambda s: s["day"])


def ols_slope(x, y):
    fit = sm.OLS(np.asarray(y, float), sm.add_constant(np.asarray(x, float))).fit()
    return fit.params[1], fit.tvalues[1], fit.pvalues[1], int(fit.nobs)


def show_t(label, res):
    print(f"{label}: t = {res.statistic:.3f}, p = {res.pvalue:.4g}, df = {res.df:.0f}")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
info = pd.DataFrame([{
    "subject": s["subject"], "day": s["day"], "session": s["session"], "task": s["description"],
    "genotype": s["genotype"], "sex": s["sex"], "subject_id": s["subject_id"], "dff": s["has_dff"],
    "event_unit": s["event_unit"], "dff_unit": s.get("dff_unit"), "size_MB": s["size"] / 1e6,
    "last_event_s": s["time"].max()} for s in sessions])
print("Sessions by subject and task (session_description):")
print(pd.crosstab(info.subject, info.task).to_string())
print("\nGenotype, sex and subject_id per subject:")
print(info.groupby("subject")[["genotype", "sex", "subject_id"]].first().to_string())
print("\nSubjects:", info.subject.nunique(), "; files with dF/F:", int(info.dff.sum()),
      "; files without:", int((~info.dff).sum()), "(subjects", sorted(info[~info.dff].subject.unique()), ")")
print("Stated units: event times", info.event_unit.unique().tolist(), "; dF/F", info.dff_unit.dropna().unique().tolist())

The archive counts 28 subjects; the files hold 28 (8 of Experiments 1 to 6, 14 of Experiment 7,
6 of Fig S8L), with the session tasks the paper describes. The Fig S8L files hold no photometry,
which that panel does not use. The Experiment 7 subject `HJ-FP-datHT-stGtACR-M7` is named as a
DAT-Cre heterozygote but its genotype field says `wt`; the code's lists treat it as DAT-Cre
(`dathetlist`), and so does this analysis. The sessions of datHT-M6 and datHT-M7 lack the task in
their file names, but their `session_description` gives it.

In [ ]:
# Event times: the files say milliseconds. The random reward task has a mean inter-reward
# interval of 12 s (Methods); the long-ITI group has 6, 12 and 30 s by name.
rows = []
for s in sessions:
    rw = jeong.events(s, jeong.BGD_REWARD)
    if s["description"] in ("RandomRewards",) or s["subject"].startswith("DB-"):
        if len(rw) > 10:
            label = s["session"].split("-")[1] if s["subject"].startswith("DB-") else "12 (Methods)"
            rows.append({"group": "Fig S8L" if s["subject"].startswith("DB-") else "random rewards",
                         "nominal_IRI": label, "mean_IRI_in_file_units": np.mean(np.diff(rw))})
iri = pd.DataFrame(rows)
print(iri.groupby(["group", "nominal_IRI"]).mean_IRI_in_file_units.describe()[["count", "mean", "min", "max"]].round(2).to_string())
print("\nSession length (last event, file units): ", info.last_event_s.describe()[["min", "50%", "max"]].round(0).to_dict())

The mean interval between random rewards is about 12 in the file's units, and 6, 12 and 30 in the
sessions named for those intervals, and sessions last 481 to 4810 units: event times are in
seconds, not the milliseconds the files state. The analysis treats them as seconds (the paper's
Python scripts do the same). This is recorded as a finding.

In [ ]:
# Photometry sampling and coverage.
rows = []
for s in sessions:
    if not s["has_dff"]:
        continue
    t = s["t"]
    rows.append({"subject": s["subject"], "session": s["session"],
                 "experiment": "7" if "stGtACR" in s["subject"] else "1-6",
                 "samples": len(t), "dt_ms": round(float(np.nanmedian(np.diff(t))) * 1000, 2),
                 "nan_time": int(np.isnan(t).sum()), "nan_dff": int(np.isnan(s["dff"]).sum()),
                 "dff_start": float(np.nanmin(t)), "dff_end": float(np.nanmax(t)),
                 "first_event": float(s["time"].min()), "last_event": float(s["time"].max()),
                 "dff_p1": float(np.nanpercentile(s["dff"], 1)), "dff_p99": float(np.nanpercentile(s["dff"], 99)),
                 "raw_samples": s.get("raw_n"), "raw_t_last": s.get("raw_t_last")})
ph = pd.DataFrame(rows)
print("Median dF/F sample interval (ms) by experiment:")
print(ph.groupby(["experiment", "dt_ms"]).size().to_string())
print("\nNaN timestamps and dF/F values per session:")
print(ph.groupby("experiment")[["nan_time", "nan_dff"]].describe().round(0).to_string())
print("\ndF/F 1st and 99th percentiles over sessions (stated unit %):",
      ph.dff_p1.median().round(2), ph.dff_p99.median().round(2))
cover = ph[(ph.dff_start > ph.first_event + 1) | (ph.dff_end < ph.last_event - 1)]
print(f"\nSessions whose dF/F starts after the first event or ends before the last one (by more than 1 s): {len(cover)}")
print(cover[["subject", "session", "dff_start", "dff_end", "first_event", "last_event"]].round(1).to_string(index=False))
raw7 = ph[ph.experiment == "7"]
rate = raw7.raw_samples / raw7.raw_t_last
big = raw7[rate > 1000]
print(f"\nExperiment 7 raw photometry (acquisition/photometry): {len(raw7)} files; {len(big)} have more than 1000 samples "
      f"per stated second (median {rate[rate > 1000].median():.0f}), their last raw timestamp {big.raw_t_last.min():.2f} to "
      f"{big.raw_t_last.max():.2f} against dF/F ending at {big.dff_end.min():.0f} to {big.dff_end.max():.0f} s; "
      f"the other {len(raw7) - len(big)} run at {rate[rate <= 1000].median():.0f} samples per second")

dF/F is sampled every 8 or 9 ms in Experiments 1 to 6 (120 Hz Doric demodulated output in the
Methods) and every 10 ms in Experiment 7, where dF/F starts before the event log (by about 15 s
in most sessions) and has no NaN timestamps. In Experiments 1 to 6 the first 20 or so timestamps
and about 300 dF/F values per session are NaN, the samples before the photometry starts; these are
left out by the measures (as `nanmean` and `min` do in the code). The sessions listed lose the end
or the start of their photometry (WT-M2 Day6 has none for its first 1117 s); the measures treat trials
there as the paper's code does. `aucsignal2event.m` (Experiment 1) sums with `sum`, so a reward
with a NaN sample in its window gets no value and is dropped. `aucsignal.m` (Experiments 2 to 6)
sums with `nansum`, so a trial outside the coverage gets an AUC of 0, and one at its edge, or near
an isolated NaN sample spread by the smoothing, the sum of its other samples. These trials are
counted under each claim, with the result when they are left out. In Experiment 7,
`calculate_auc` sums the samples present, and a window with none gets no value. In 65 of the 101 Experiment 7 files
the raw photometry holds about 12 million samples per stated second and ends at a timestamp of
0.43 to 4.18, where dF/F runs to 596 to 4133 s: the timestamps of the raw series are in thousands
of seconds (12 kHz, the Doric detection rate of the Methods). The raw series is not used here; this is recorded as a finding.

In [ ]:
# dF/F scale: the files say %. The Methods define dF/F as a fraction, (470 - fitted 405) / fitted
# 405. Compare the stored dF/F with that fraction computed from the raw channels of one session
# per photometry system (the paper's baseline correction is not repeated, so only the scale and
# the correlation are checked).
import h5py, remfile
check = [a for a in assets if a["path"].endswith("sub-HJ-FP-M2_ses-Day1.nwb")
         or a["path"].endswith("sub-HJ-FP-WT-stGtACR-F3_ses-Day2-Pavlovian.nwb")]
for a in check:
    with h5py.File(remfile.File(a["url"]), "r") as f:
        raw = f["acquisition/photometry/data"][:]
        rt = f["acquisition/photometry/timestamps"][:]
        d = f["processing/photometry/dff/data"][:]
        dt = f["processing/photometry/dff/timestamps"][:]
    good = np.isfinite(rt) & np.isfinite(raw).all(1)
    best = None
    for sig, ref in [(0, 1), (1, 0)]:
        c = np.polyfit(raw[good, ref], raw[good, sig], 1)
        fit = np.polyval(c, raw[good, ref])
        frac = (raw[good, sig] - fit) / fit
        # dF/F timestamps are on the behaviour clock; the raw series starts at its own 0. Map the
        # raw samples onto dF/F time by their position in the recording.
        td = dt[np.isfinite(dt)]
        pos = (rt[good] - rt[good][0]) / (rt[good][-1] - rt[good][0])
        stored = np.interp(td[0] + pos * (td[-1] - td[0]), td, d[np.isfinite(dt)])
        ok = np.isfinite(stored)
        r = np.corrcoef(frac[ok], stored[ok])[0, 1]
        slope = np.polyfit(frac[ok], stored[ok], 1)[0]
        if best is None or r > best[1]:
            best = (sig, r, slope)
    print(f"{a['path']}: channel {best[0]} as the signal, r = {best[1]:.2f} with stored dF/F, "
          f"stored dF/F = {best[2]:.1f} x the fraction")

The stored dF/F follows the fraction computed from the raw channels and is about 100 times it, so
it is in percent, as stated. The paper's Matlab scripts give their bounds for fitting dopamine
responses (claim fig4hi-delay-change-dopamine-unchanged) in units of a fraction times
milliseconds (`aucsignal.m` multiplies by the bin in ms); the helper divides dF/F by 100 for the
Matlab-derived measures, so that their values are on the scale the Methods define (a fraction).
Whether the code's bounds were set for this scale is not known: they bind on it as well as on
percent (fig4hi-delay-change-dopamine-unchanged), and the analysis assumes the paper's fit was
bounded as here. Ratios, correlations and t-tests do not depend on it. For the
pyPhotometry session the raw samples are placed on the dF/F clock by their position in the
recording, since the dF/F timestamps start about 15 s before the raw series' zero.

In [ ]:
# Recordings deposited more than once: identical event logs or dF/F traces across files.
import hashlib
def digest(a):
    return hashlib.md5(np.ascontiguousarray(a).tobytes()).hexdigest()
by_events, by_dff, by_licks, by_schedule = {}, {}, {}, {}
for s in sessions:
    by_events.setdefault(digest(s["time"]), []).append(s["path"])
    by_licks.setdefault(digest(jeong.events(s, jeong.LICK)), []).append(s["path"])
    by_schedule.setdefault(digest(np.round(s["time"][np.isin(s["code"], [7, 10, 15, 16])], 2)), []).append(s["path"])
    if s["has_dff"]:
        by_dff.setdefault(digest(np.nan_to_num(s["dff"])), []).append(s["path"])
for name, d in [("event logs", by_events), ("lick times", by_licks), ("dF/F traces", by_dff),
                ("reward and cue schedules", by_schedule)]:
    dup = [v for v in d.values() if len(v) > 1]
    print(f"Files sharing identical {name}: {len(dup)} groups")
    for v in dup:
        print("   ", " = ".join(p.split("/")[1] for p in v))

No two files share an event log, lick times or a dF/F trace, so no recording is deposited twice.
Eight pairs of sessions share the same reward and cue times: the task schedules were reused
(across animals, or across days of one animal), while the licks and dF/F differ. Each file is
counted as one session.

In [ ]:
# Shared clock of dF/F and the event log. Per session, the response to unpredicted events (the
# first lick after a random reward, cue onsets) or the contrast of CS+ against CS- onsets
# (Experiments 2 to 5), with the event times shifted by lags from -2 to 2 s, and by 10 s. If the
# two streams share a clock, the response peaks near lag 0 (within the response latency).
lags = np.round(np.arange(-2, 2.01, 0.25), 2)   # the rewards come 3 s after cues; larger lags reach them
csplus_ids = {}
for m in jeong.PAV_MICE:
    first = [s for s in subject_sessions(m, "Pavlovian") if s["day"] == jeong.CUMSUM_START[m][0]][0]
    csplus_ids[m] = jeong.cs_plus_identity(first)
rows = []
for s in sessions:
    if not s["has_dff"]:
        continue
    prof, n, ref = jeong.response_profile(s, lags, csplus_ids.get(s["subject"]))
    group = "7" if "stGtACR" in s["subject"] else {"RandomRewards": "1", "Pavlovian": "2-5", "Poisson": "6"}[s["description"]]
    rows.append({"experiment": group, "reference": ref, "subject": s["subject"], "session": s["session"], "n": n,
                 "peak_lag": lags[int(np.nanargmax(prof))], "at_0": prof[lags == 0][0],
                 "shift_10s": jeong.response_profile(s, [10.0], csplus_ids.get(s["subject"]))[0][0]})
clock = pd.DataFrame(rows)
print("Lag (s) of the largest response, per session:")
print(clock.groupby(["experiment", "reference"]).peak_lag.describe()[["count", "min", "25%", "50%", "75%", "max"]].to_string())
print("\nSessions with a positive response at lag 0, and with the events shifted by 10 s:")
print(clock.groupby(["experiment", "reference"]).apply(lambda g: pd.Series({
    "sessions": len(g), "positive_at_0": int((g.at_0 > 0).sum()), "positive_shift_10s": int((g.shift_10s > 0).sum()),
    "median_at_0": g.at_0.median(), "median_shift_10s": g.shift_10s.median()})).round(3).to_string())
far = clock[(clock.peak_lag < -1) | (clock.peak_lag > 1)]
print(f"\nSessions whose peak lag is beyond 1 s from 0: {len(far)}")
print(far[["subject", "session", "reference", "n", "peak_lag", "at_0"]].round(3).to_string(index=False))

Across sessions the median lag of the largest response is 0 or -0.25 s in every group, and the
quartiles lie within 0.31 s of 0; the response is positive at lag 0 in 50 of 56, 198 of 210, 24 of 25, 79 of 87 and 14 of 14
sessions, and shifting the events by 10 s removes it (medians -0.13 to 0.01). This is consistent
with dF/F and the event log sharing one clock in every experiment, including Experiment 7, whose
dF/F timestamps start before the event log. It is not established for the 27 sessions listed,
whose peak lies more than 1 s from 0: they are sessions with no clear response to the reference
event (the first random reward session of M2, the first sessions of conditioning, the background
reward sessions where CS+ and CS- responses become similar, DAT-Cre sessions under inhibition),
where the peak is not defined by a response.

## session-day-link

> The subjects and "Day" numbers that the paper's analysis code names for each experiment are
> sessions of the dandiset, so that the code's selection of sessions can be applied to it.

Subjects are matched with `_` read as `-`, and sessions by the number after "Day" in the file name
(`load_dandi_url`). For each script's day list, the listed days (startday, endday, learnedday and
the single days of `omission.m`) are checked to be sessions of the task, and every session in each
range is checked for the condition: 2 s cue (3 s cue-reward delay) for Experiment 2, 8 s cue (9 s
delay) for Experiment 3, background rewards for Experiment 5, no delivered reward for Experiment 4,
the trial-less task (`Poisson`) for Experiment 6. The criterion's "every Day number from startday
to endday" is also counted literally, every integer day in each range.

In [ ]:
cond = {(s["subject"], s["day"]): jeong.session_condition(s) for s in sessions if s["subject"] in jeong.EXP1_MICE}
present = {(s["subject"], s["day"]): s["description"] for s in sessions}
print("Subjects of the code's lists missing from the dandiset:",
      [m for m in jeong.EXP1_MICE + jeong.DAT_MICE + [jeong.DAT_EXCLUDED] + jeong.CONTROL_MICE + jeong.LONGITI_MICE
       if m not in set(info.subject)])

def expect(c, kind):
    if c is None:
        return False
    d = c.get("delays", [])
    return {"2s": c["task"] == "Pavlovian" and d == [3.0] and c["n_bgd"] == 0 and c["delivered"] > 0,
            "8s": c["task"] == "Pavlovian" and 9.0 in d and c["n_bgd"] == 0 and c["delivered"] > 0,
            "8s_or_2s": c["task"] == "Pavlovian" and c["n_bgd"] == 0 and c["delivered"] > 0,
            "bgd": c["task"] == "Pavlovian" and c["n_bgd"] > 0,
            "ext": c["task"] == "Pavlovian" and c.get("delivered", 1) == 0,
            "poisson": c["task"] == "Poisson"}[kind]

checks = []   # (script, subject, kind, day, listed day?, present, condition ok)
def add_range(script, m, kind, a, b, listed=()):
    for d in range(a, b + 1):
        c = cond.get((m, d))
        checks.append({"script": script, "subject": m, "condition": kind, "day": d,
                       "listed": d in listed, "session": c is not None,
                       "condition_ok": expect(c, kind) if c is not None else np.nan})

for m in jeong.PAV_MICE:
    st, en = jeong.CUMSUM_START[m], jeong.CUMSUM_END[m]
    add_range("cumsum_pavlovian C1 (Exp 2)", m, "2s", st[0], en[0], {st[0], en[0], jeong.LEARNED_DAY[m]})
    add_range("cumsum_pavlovian C2 pre (Exp 2)", m, "2s", st[1] - 3, st[1] - 1)
    add_range("cumsum_pavlovian C2 (Exp 3)", m, "8s", st[1], en[1], {st[1], en[1]})
    add_range("cumsum_pavlovian C3 pre (Exp 3)", m, "8s", st[2] - 1, st[2] - 1)
    add_range("cumsum_pavlovian C3 (Exp 4)", m, "ext", st[2], en[2], {st[2], en[2]})
    add_range("cumsum_pavlovian C4 (Exp 5)", m, "bgd", st[3], en[3], {st[3], en[3]})
    bs, be = jeong.BGD_START[m], jeong.BGD_END[m]
    add_range("cumsum_bgdrw_vs_extinction (Exp 5)", m, "bgd", bs[0], be[0], {bs[0], be[0]})
    add_range("cumsum_bgdrw_vs_extinction (Exp 4)", m, "ext", bs[1], be[1], {bs[1], be[1]})
    add_range("intermediate_rsp_trialless (Exp 6)", m, "poisson", jeong.TRIALLESS_LEARNED[m], jeong.TRIALLESS_END[m],
              {jeong.TRIALLESS_LEARNED[m], jeong.TRIALLESS_END[m]})
    o3, ox = jeong.OMISSION_DAYS[m]
    add_range("omission (Exp 3 first)", m, "8s", o3, o3, {o3})
    add_range("omission (Exp 4 first)", m, "ext", ox, ox, {ox})
ck = pd.DataFrame(checks)
listed = ck[ck.listed]
print(f"Listed days (start, end, learned, single days): {len(listed)}; sessions: {int(listed.session.sum())}; "
      f"with the condition: {int(listed.condition_ok.fillna(False).astype(bool).sum())}")
print(listed[~listed.session | (listed.condition_ok == False)].to_string(index=False))
sess = ck[ck.session]
print(f"\nSessions in the ranges: {len(sess)}; with the condition: {int(sess.condition_ok.astype(bool).sum())}")
print(sess[sess.condition_ok == False].to_string(index=False))
print(f"\nEvery integer day of the ranges: {len(ck)}; of them sessions: {int(ck.session.sum())} "
      f"({ck.session.mean():.1%})")
print("Integer days in the ranges with no session:")
print(ck[~ck.session].groupby(["subject"]).day.apply(lambda d: sorted(set(d))).to_string())

In [ ]:
# The mouse and day lists as the scripts define them, read from the repository at the pinned
# commit, against the copies in jeong.py used above.
code_lists = jeong.fetch_code_lists()
col = lambda d, c: [d[m][c] for m in jeong.PAV_MICE]
mine = {
    "analysis/fig4/cumsum_pavlovian.m": {
        "startday": [jeong.CUMSUM_START[m] for m in jeong.PAV_MICE], "endday": [jeong.CUMSUM_END[m] for m in jeong.PAV_MICE],
        "learnedday": [[jeong.LEARNED_DAY[m] for m in jeong.PAV_MICE]], "ndaybeforestart": [jeong.CUMSUM_BEFORE],
        "ratio": [jeong.CUMSUM_RATIO]},
    "analysis/fig4/cumsum_bgdrw_vs_extinction.m": {
        "startday": [jeong.BGD_START[m] for m in jeong.PAV_MICE], "endday": [jeong.BGD_END[m] for m in jeong.PAV_MICE]},
    "analysis/fig5/intermediate_rsp_trialless.m": {
        "learnedday": [[jeong.TRIALLESS_LEARNED[m] for m in jeong.PAV_MICE]], "endday": [[jeong.TRIALLESS_END[m] for m in jeong.PAV_MICE]]},
    "analysis/figS10/omission.m": {"startday": [col(jeong.OMISSION_DAYS, 0), col(jeong.OMISSION_DAYS, 1)]},
    "analysis/figS10/cue_acquisition.m": {
        "startday": [col(jeong.CUMSUM_START, 0)], "learnedday": [[jeong.LEARNED_DAY[m] for m in jeong.PAV_MICE]],
        "endday": [col(jeong.CUMSUM_END, 0)]},
    "analysis/fig6/backpropagation_pavlovian.m": {"startday": [col(jeong.CUMSUM_START, 0)], "endday": [col(jeong.CUMSUM_END, 0)]},
}
all_match = True
for path, found in code_lists.items():
    print(path)
    for name, (line, value) in found.items():
        if name == "mouseList":
            ok = value == jeong.PAV_MICE
        else:
            ok = np.array_equal(value, np.array(mine[path][name], float))
        all_match &= bool(ok)
        print(f"  {'matches' if ok else 'DIFFERS'}: {line}")
    missing = set(mine[path]) - set(found)
    if missing:
        all_match = False
        print("  not found in the script:", missing)
print("Every list in jeong.py equals the script's:", all_match)

In [ ]:
# The 8 s condition's first session in two animals holds both delays (the cue lengthened within
# the session); the condition of each range's sessions, by delays:
for m in jeong.PAV_MICE:
    st, en = jeong.CUMSUM_START[m][1], jeong.CUMSUM_END[m][1]
    print(m, {d: cond[(m, d)].get("delays") for d in range(st, en + 1) if (m, d) in cond})

**Result: partially reproduced.** All 28 subjects of the code's lists are in the dandiset. Of the
119 listed days, 117 are sessions with the condition; the exception is HJ-FP-M4 Day38 (listed twice), the
first day of Experiment 5 in both background reward scripts, which is not in the dandiset (its
next session, Day39, has background rewards). All 320 sessions inside the ranges have the condition the
script names. Counted literally, every integer day of the ranges, 320 of 331
(96.7%) are sessions; the 7 missing days within the Pavlovian ranges (M2 Day23, M3 Day16 and Day31, M4 Day30
and Day38, F2 Day16, M6 Day49) fall where the paper's session counts imply a session (see
methods-session-counts). The subjects match and at least 90% of the listed days do, so the link
holds in part. The claims that depend on it are tested on the sessions that are present. The mouse and day lists
in `jeong.py` equal those the six scripts define at the pinned commit (second output); the
start, end and learned days of cue_acquisition.m and backpropagation_pavlovian.m equal the
initial learning columns of cumsum_pavlovian.m.

## methods-session-counts

> Animals were trained for 7 +/- 2.1 sessions of random rewards, 13.3 +/- 2.7 sessions of initial
> conditioning, 4.3 +/- 0.8 sessions with the 8 s cue, 8.1 +/- 2.5 sessions with background
> rewards and 2.6 +/- 0.8 sessions of extinction; in sequential conditioning, 5.6 +/- 2.4 (wild
> type) and 7.1 +/- 0.7 (DAT-Cre) sessions (mean +/- SD).

Sessions are counted from the dandiset per animal. Experiment 1, the random reward sessions of the
8 mice. Experiments 2, 3 and 4, the sessions in the code's ranges (cumsum_pavlovian.m, startday to
endday of conditions 1 to 3). Experiment 5, the code gives two ranges, condition 4 of
cumsum_pavlovian.m and cumsum_bgdrw_vs_extinction.m; both are counted, and also the sessions with
background rewards in the event log, which is the count from the data. Experiment 7, the
conditioning sessions after the random reward session, all 7 mice of each group as in the Methods.
The counts of integer days in each range are shown beside them.

In [ ]:
counts = []
for m in jeong.EXP1_MICE:
    counts.append({"quantity": "Exp 1", "subject": m, "sessions": len(subject_sessions(m, "RandomRewards"))})
for m in jeong.PAV_MICE:
    days = {s["day"] for s in subject_sessions(m, "Pavlovian")}
    st, en = jeong.CUMSUM_START[m], jeong.CUMSUM_END[m]
    for q, c in [("Exp 2", 0), ("Exp 3", 1), ("Exp 4", 2), ("Exp 5 (cumsum_pavlovian C4)", 3)]:
        rng_ = range(st[c], en[c] + 1)
        counts.append({"quantity": q, "subject": m, "sessions": len(days & set(rng_)), "integer_days": len(rng_)})
    rng_ = range(jeong.BGD_START[m][0], jeong.BGD_END[m][0] + 1)
    counts.append({"quantity": "Exp 5 (cumsum_bgdrw range)", "subject": m, "sessions": len(days & set(rng_)), "integer_days": len(rng_)})
    counts.append({"quantity": "Exp 5 (sessions with background rewards)", "subject": m,
                   "sessions": sum(cond[(m, d)]["n_bgd"] > 0 for d in days)})
for grp, mice in [("Exp 7 WT", jeong.CONTROL_MICE), ("Exp 7 DAT-Cre", jeong.DAT_MICE + [jeong.DAT_EXCLUDED])]:
    for m in mice:
        ss = subject_sessions(m)
        assert ss[0]["description"] == "RandomRewards" and all(s["description"] == "Pavlovian" for s in ss[1:])
        counts.append({"quantity": grp, "subject": m, "sessions": len(ss) - 1})
counts = pd.DataFrame(counts)
summary = counts.groupby("quantity", sort=False).agg(animals=("subject", "size"), mean=("sessions", "mean"),
                                                      sd=("sessions", "std"), min=("sessions", "min"),
                                                      max=("sessions", "max"), integer_day_mean=("integer_days", "mean"))
paper = {"Exp 1": (7, 2.1), "Exp 2": (13.3, 2.7), "Exp 3": (4.3, 0.8), "Exp 5 (cumsum_bgdrw range)": (8.1, 2.5),
         "Exp 5 (sessions with background rewards)": (8.1, 2.5), "Exp 5 (cumsum_pavlovian C4)": (8.1, 2.5),
         "Exp 4": (2.6, 0.8), "Exp 7 WT": (5.6, 2.4), "Exp 7 DAT-Cre": (7.1, 0.7)}
summary["paper"] = [paper[q][0] for q in summary.index]
summary["tolerance"] = [paper[q][1] for q in summary.index]
summary["within"] = (summary["mean"] - summary.paper).abs() <= summary.tolerance
print(summary.round(2).to_string())

**Result: reproduced.** Counted from the dandiset, the means are 7.0 random reward sessions
(4 to 11, SD 2.14), 12.7 of Experiment 2, 4.3 of Experiment 3, 7.86 of Experiment 5 (sessions with
background rewards and the cumsum_bgdrw range; 7.14 with condition 4 of cumsum_pavlovian.m),
2.43 of extinction, and 5.43 and 7.0 sequential conditioning sessions for wild type and DAT-Cre.
All seven are within the stated SDs. The counts of integer days in the code's ranges, 13.3, 8.1
(cumsum_bgdrw) and 2.6, equal the paper's means, which suggests that the paper counted sessions
that are not in the dandiset (session-day-link). The SDs of the Experiment 7 counts, 0.79 for wild type and 2.45 for DAT-Cre (from datHT-M7's
12 sessions), are close to the paper's 2.4 and 0.7 taken in the other order, so the paper may
have swapped them; the criterion tests only the means.

## Experiment 1: random rewards

The 12 s random reward sessions of the 8 mice, in Day order (the dandiset holds no session named
"30s", so the code's cut leaves all of them). The measures follow corr_randomrewards.m as the
criteria give it; see `jeong.exp1_rewards`.

In [ ]:
exp1 = {}
for m in jeong.EXP1_MICE:
    exp1[m] = [(s, jeong.exp1_rewards(s)) for s in subject_sessions(m, "RandomRewards")]
rows = []
for m, lst in exp1.items():
    for k, (s, r) in enumerate(lst, start=1):
        keep = ~r["out"]
        ok = keep & np.isfinite(r["iri"]) & np.isfinite(r["response"])
        rows.append({"subject": m, "session": k, "day": s["day"], "rewards": len(r["reward"]),
                     "short": r["short"].mean(), "nolick": r["nolick"].mean(),
                     "kept": int(keep.sum()), "kept_with_dff": int(np.isfinite(r["response"][keep]).sum()),
                     "r_iri": stats.pearsonr(r["iri"][ok], r["response"][ok])[0]})
exp1_sessions = pd.DataFrame(rows)
print(exp1_sessions.groupby("subject")[["rewards", "kept", "kept_with_dff"]].sum().to_string())

## exp1-excluded-rewards

> In Experiment 1, 22.6 +/- 2.4% of rewards came less than 3 s after the previous reward and 5.7
> +/- 0.1% had no lick before the next reward; both were excluded.

In [ ]:
per_animal = exp1_sessions.groupby("subject")[["short", "nolick"]].mean() * 100
print("Per animal (mean over sessions, %):")
print(per_animal.round(2).to_string())
a = per_animal.mean()
b = exp1_sessions[["short", "nolick"]].mean() * 100
print(f"\nReading (a), mean over animals: short IRI {a['short']:.2f}% (SD over animals {per_animal['short'].std():.2f}), "
      f"no lick {a['nolick']:.2f}% (SD {per_animal['nolick'].std():.2f})")
print(f"Reading (b), mean over {len(exp1_sessions)} sessions: short IRI {b['short']:.2f}% "
      f"(SD {exp1_sessions['short'].std() * 100:.2f}), no lick {b['nolick']:.2f}% (SD {exp1_sessions['nolick'].std() * 100:.2f})")
print("Paper: 22.6 +/- 2.4 and 5.7 +/- 0.1")

**Result: reproduced.** Under reading (a) 22.59% of rewards follow the previous one by less than
3 s and 5.66% have no lick before the next reward; under reading (b) 22.59% and 5.68%. Both
values are within tolerance under both readings. The spreads printed (SD over animals 2.39 and
1.04) suggest the paper's 2.4 is the SD over animals of the first; its 0.1 matches neither.

## fig3e-reward-response-increases

> In experimentally naive mice given unpredicted sucrose, the dopamine response to reward increases
> with the number of rewards experienced, in every animal (Test 1).

In [ ]:
r_trial = {}
for m, lst in exp1.items():
    n, resp, num = 0, [], []
    for s, r in lst:
        keep = ~r["out"]
        resp.append(r["response"][keep])
        num.append(np.flatnonzero(keep) + n)
        n += len(r["reward"])
    R, N = np.concatenate(resp), np.concatenate(num)
    ok = np.isfinite(R)
    r_trial[m] = stats.pearsonr(N[ok], R[ok])[0]
r_trial = pd.Series(r_trial)
print("Per animal Pearson r, response against reward number:")
print(r_trial.round(3).to_string())
print(f"mean r {r_trial.mean():.3f}; animals with r > 0: {(r_trial > 0).sum()} of {len(r_trial)}")
show_t("one-sample t-test against 0", stats.ttest_1samp(r_trial, 0))
print("Paper: t(7) = 4.40, p = 0.0031")

**Result: reproduced.** The correlation between the reward response and the reward number is
positive in all 8 animals (0.068 to 0.677, mean 0.308), t(7) = 4.41, p = 0.0031, the paper's
values.

## fig3g-response-iri-positive

> The dopamine response to an unpredicted reward is positively correlated with the interval since
> the previous reward (Test 2).

In [ ]:
r_iri = exp1_sessions.groupby("subject").r_iri.mean()
print("Per animal mean over sessions of the per-session r (response against previous IRI):")
print(r_iri.round(3).to_string())
show_t("one-sample t-test against 0", stats.ttest_1samp(r_iri, 0))
print("Paper: t(7) = 5.95, p = 5.7e-4")

**Result: reproduced.** The mean per-session correlation with the previous IRI is positive in all
8 animals (0.029 to 0.405), t(7) = 5.95, p = 5.7e-4, the paper's values.

## figs8k-iri-correlation-stable

> The positive correlation between reward response and previous IRI holds across sessions and does
> not change over the first 7 sessions.

In [ ]:
first7 = exp1_sessions[exp1_sessions.session <= 7]
slope, t, p, n = ols_slope(first7.session, first7.r_iri)
print(f"Linear regression of per-session r on session number (1 to 7), n = {n} sessions: slope {slope:.4f}, t = {t:.2f}, p = {p:.4f}")
print("Paper: t = -0.04, p = 0.9707, n = 50")
print("\nMean r over animals per session (and animals):")
print(first7.groupby("session").r_iri.agg(["mean", "size"]).round(3).to_string())

**Result: reproduced.** Over the 50 sessions the slope of r on session number is -0.0005 per
session, t = -0.04, p = 0.97 (paper t = -0.04, p = 0.9707), and the mean r over animals is
positive in each of sessions 1 to 7 (0.134 to 0.288).

## figs8b-lick-rates-constant

> Total, consummatory and non-consummatory lick rates do not change across the first 7 random
> reward sessions.

In [ ]:
lick_rows, lick_rw = [], {}
for m, lst in exp1.items():
    L, R = [], []
    for k, (s, r) in enumerate(lst, start=1):
        lk = jeong.exp1_licks(s)
        lick_rows.append({"subject": m, "session": k, **lk["rates"]})
        L.append(lk["lick_rate_rw"]); R.append(lk["response_rw"])
        exp1[m][k - 1] = (s, r, lk)
    lick_rw[m] = (np.concatenate(L), np.concatenate(R))
licks = pd.DataFrame(lick_rows)
l7 = licks[licks.session <= 7]
for c in ["total", "consummatory", "nonconsummatory"]:
    slope, t, p, n = ols_slope(l7.session, l7[c])
    print(f"{c:16s} rate (licks/s): mean {l7[c].mean():.2f}, slope {slope:.4f} per session, t = {t:.2f}, p = {p:.4f}, n = {n}")
print("Paper: t = -0.79, -0.20, -0.58; p = 0.4389, 0.8397, 0.5618")

**Result: reproduced.** None of the three lick rates changes with session over the 50 sessions:
t = -0.78 (p = 0.4389), -0.20 (p = 0.8397) and -0.58 (p = 0.5618), matching Table S1.

## figs8d-consummatory-lick-uncorrelated

> The consummatory lick rate after a reward is not correlated with the dopamine response to that
> reward.

In [ ]:
rc = {}
for m, (L, R) in lick_rw.items():
    ok = np.isfinite(L) & np.isfinite(R)
    rc[m] = (stats.pearsonr(L[ok], R[ok])[0], int(ok.sum()), int((~np.isfinite(L)).sum()))
rc = pd.DataFrame(rc, index=["r", "rewards", "dropped_inf_or_nan"]).T
print(rc.round(3).to_string())
print(f"mean r {rc.r.mean():.3f}")
show_t("one-sample t-test against 0", stats.ttest_1samp(rc.r, 0))
print("Paper: t = 0.23, p = 0.8245")

**Result: reproduced.** The mean correlation is 0.010, t(7) = 0.23, p = 0.82 (paper t = 0.23,
p = 0.8245): no correlation is found.

## figs8e-lick-bout-responses

> Dopamine rises at the onset of consummatory lick bouts but falls slightly at the onset of
> non-consummatory (dry) lick bouts, so the rising reward response is not a general response to
> licking.

In [ ]:
windows = {"a: 0 to 1 s": (0.0, 1.0), "b: -0.5 to 1 s": (-0.5, 1.0), "c: 0 to 0.5 s": (0.0, 0.5)}
zrows = []
for m, lst in exp1.items():
    for k, (s, r, lk) in enumerate(lst, start=1):
        z = jeong.bout_onset_z(s, lk, windows)
        zrows.append({"subject": m, "session": k, **{f"{a}|{b}": v for (a, b), v in z.items()}})
zdf = pd.DataFrame(zrows)
za = zdf.groupby("subject").mean(numeric_only=True)
print("Onsets per session (median): consummatory", zdf["consummatory|n"].median(), "; non-consummatory", zdf["nonconsummatory|n"].median())
for w in windows:
    c, n = za[f"consummatory|{w}"], za[f"nonconsummatory|{w}"]
    print(f"\nReading {w}: consummatory mean z {c.mean():.3f}, non-consummatory {n.mean():.3f} "
          f"(non-consummatory below 0 in {(n < 0).sum()} of 8 animals)")
    show_t("  paired, consummatory against non-consummatory", stats.ttest_rel(c, n))
    show_t("  consummatory against 0", stats.ttest_1samp(c, 0))
    show_t("  non-consummatory against 0", stats.ttest_1samp(n, 0))
print("\nPaper: paired t = 3.28, p = 0.0134; consummatory t = 3.03, p = 0.0191; non-consummatory t = -2.55, p = 0.0379")

**Result: blocked (methods_underspecified).** Under every reading the consummatory onset response
is positive (mean z 1.76 to 3.07, p <= 4.0e-4) and above the non-consummatory one (p <= 1.8e-4).
The non-consummatory response is negative on average under all three readings, but significantly
so only from 0 to 0.5 s (reading c, z = -0.35, t = -2.56, p = 0.0376, close to the paper's -2.55,
p = 0.0379); from 0 to 1 s and from -0.5 to 1 s it is -0.15 (p = 0.14) and -0.02 (p = 0.80). The
window is fixed by neither the Methods, the legend nor the code, and the readings give different
outcomes, so under the claims file's rule for readings of an open method detail the claim is
blocked. The consummatory t values against 0 (6.30 to 6.49) are about twice the paper's 3.03, so the paper's
measure of the response differs from these readings in a way not found here.

## figs8g-baseline-unchanged

> Spontaneous baseline dopamine events, away from rewards and licks, do not change in size across
> random reward sessions.

In [ ]:
brows = []
for m, lst in exp1.items():
    thr = jeong.peak_threshold(lst[0][0])
    for k, (s, r, lk) in enumerate(lst, start=1):
        bp = jeong.baseline_peaks(s, thr)
        brows.append({"subject": m, "session": k, "threshold": thr,
                      "reward_response": np.nanmean(r["response"][~r["short"]]), **bp})
base = pd.DataFrame(brows)
print("Peaks per session (all above threshold, kept by the Methods' rule, kept by the code's rule):")
print(base[["n_peaks", "n_methods", "n_code"]].describe().loc[["min", "50%", "max"]].to_string())
bcorr = base.groupby("subject").apply(lambda g: pd.Series({
    "r_methods": stats.pearsonr(g.session, g.methods)[0], "r_code": stats.pearsonr(g.session, g["code"])[0],
    "r_reward": stats.pearsonr(g.session, g.reward_response)[0], "sessions": len(g)}))
print(bcorr.round(3).to_string())
for c in ["r_methods", "r_code"]:
    print(f"{c}: mean r {bcorr[c].mean():.3f}"); show_t("  one-sample t-test against 0", stats.ttest_1samp(bcorr[c], 0))
print("Paper: t = 0.11, p = 0.9190")

**Result: reproduced.** With peaks kept by the Methods' rule, the mean correlation of peak size
with session is -0.034, t(7) = -0.13, p = 0.90; with the code's rule, t(7) = 0.07, p = 0.95 (paper
t = 0.11, p = 0.919). No change is found. Per animal the correlations range from -0.98 to 0.97.

## figs8h-baseline-uncorrelated-with-reward-change

> Across animals, the change of baseline dopamine activity over sessions is not correlated with the
> change of the reward response over sessions.

In [ ]:
for c in ["r_methods", "r_code"]:
    rr, pp = stats.pearsonr(bcorr[c], bcorr.r_reward)
    print(f"baseline {c} against reward-response r over 8 animals: r = {rr:.3f}, p = {pp:.3f}")
print("reward-response r per animal:", bcorr.r_reward.round(3).tolist())

**Result: reproduced.** Across the 8 animals, the session correlation of baseline peak size is not
correlated with that of the reward response (Methods' rule r = -0.10, p = 0.81; code's rule
r = -0.09, p = 0.83), while the reward response rises with session in every animal (r 0.23 to
0.92).

## figs8l-lick-rate-tracks-iri

> In a separate group of 6 mice, the non-consummatory lick rate depends on the session's mean IRI
> (6, 12 or 30 s) while the consummatory lick rate does not, so animals learn the average reward
> rate within at most two sessions.

The IRI of each session is read from its name (`Day1-30` is 30 s), and checked against the
intervals in the data check above.

In [ ]:
lrows = []
for m in jeong.LONGITI_MICE:
    for s in subject_sessions(m):
        lrows.append({"subject": m, "day": s["day"], "iri": int(s["session"].split("-")[1]), **jeong.longiti_rates(s)})
lr = pd.DataFrame(lrows)
print(pd.crosstab(lr.subject, lr.iri).to_string())
g = lr.groupby(["subject", "iri"])[["consummatory", "nonconsummatory"]].mean().reset_index()
g["rank"] = g.iri.map({6: 1, 12: 2, 30: 3})
print(g.groupby("iri")[["consummatory", "nonconsummatory"]].mean().round(3).to_string())
for reg in ["iri", "rank"]:
    for c in ["nonconsummatory", "consummatory"]:
        slope, t, p, n = ols_slope(g[reg], g[c])
        print(f"reading {'(a) IRI in s' if reg == 'iri' else '(b) rank'}: {c} slope {slope:.4f}, t = {t:.2f}, p = {p:.4f}, n = {n}")
print("Paper: non-consummatory t = -2.73, p = 0.0147; consummatory t = 1.00, p = 0.3338")

**Result: reproduced.** With the IRI in seconds the non-consummatory rate falls with the IRI
(t = -2.73, p = 0.0147) and the consummatory rate does not change (t = 1.00, p = 0.334), the
paper's values; with the rank of the condition, t = -2.61 (p = 0.019) and t = 1.34 (p = 0.199).

## Experiments 2 to 5: Pavlovian conditioning

Per animal, the CS+ is the cue whose reward is delivered in the first session of Experiment 2.
Per CS+ trial `jeong.cs_plus_trials` computes the dopamine and lick measures of the criteria.

In [ ]:
pav = {}
csplus = {}
for m in jeong.PAV_MICE:
    ss = subject_sessions(m, "Pavlovian")
    first = [s for s in ss if s["day"] == jeong.CUMSUM_START[m][0]][0]
    csplus[m] = jeong.cs_plus_identity(first)
    pav[m] = {s["day"]: jeong.cs_plus_trials(s, csplus[m]) for s in ss}
unpaired = [(s["subject"], s["day"]) for s in sessions if s["subject"] in jeong.PAV_MICE and s["description"] == "Pavlovian"
            and not jeong.pavlovian_events(s)["paired"]]
print("CS+ identity (1 = code 15, 2 = code 16):", csplus)
print("Sessions where cues and reward events do not pair one to one:", unpaired)
nan_trials = sum(int(np.isnan(v["da"]).sum()) for m in pav for v in pav[m].values())
inc_trials = sum(int(v["da_incomplete"].sum()) for m in pav for v in pav[m].values())
print("CS+ trials:", sum(len(v["da"]) for m in pav for v in pav[m].values()), "; without a dopamine value:", nan_trials,
      "; with a NaN smoothed dF/F sample from -1 to 2 s (summed by nansum, as aucsignal.m does):", inc_trials)
R = jeong.matlab_round


def cumsum_condition(m, c, drop_incomplete=False):
    # cumsum_pavlovian.m for condition c (0 initial learning, 1 cue duration, 2 extinction).
    # With drop_incomplete, trials with a NaN smoothed dF/F sample from -1 to 2 s are left out of
    # the series (a variant; the code keeps them with their nansum AUC).
    st, en = jeong.CUMSUM_START[m][c], jeong.CUMSUM_END[m][c]
    days = [d for d in sorted(pav[m]) if st - jeong.CUMSUM_BEFORE[c] <= d <= en]
    cut = jeong.LEARNED_DAY[m] if c == 0 else st
    out = {"days": days}
    keep = {d: np.ones(len(pav[m][d]["da"]), bool) for d in days}
    if c == 1:
        keep = {d: np.isfinite(pav[m][d]["lick_median_ms"]) for d in days}
        out["dropped_no_lick"] = sum(int((~keep[d]).sum()) for d in days)
    if drop_incomplete:
        keep = {d: keep[d] & ~pav[m][d]["da_incomplete"] for d in days}
    cat = lambda f: np.concatenate([f(pav[m][d])[keep[d]] for d in days])
    nb = sum(int(keep[d].sum()) for d in days if d < cut)
    da = cat(lambda v: v["da"])
    if c == 1:
        out["lick_series"] = {"median lick time": cat(lambda v: v["lick_median_ms"])}
    else:
        out["lick_series"] = {"Methods 0-2 s": cat(lambda v: v["lick_methods"]),
                              "code cue+trace": cat(lambda v: v["lick_code_cs"] - v["lick_code_base"])}
    nt = min(R(nb * jeong.CUMSUM_RATIO[c]), len(da))   # the cap binds only when trials are left out
    out.update(nb=nb, nt=nt, da=da, n_incomplete=int(cat(lambda v: v["da_incomplete"])[:nt].sum()))
    out["da_abrupt"], out["da_change"] = jeong.abruptness(np.nancumsum(da), nt)
    out["lick"] = {k: jeong.abruptness(np.cumsum(v), nt) for k, v in out["lick_series"].items()}
    return out


def abrupt_table(c, drop_incomplete=False):
    rows = []
    for m in jeong.PAV_MICE:
        o = cumsum_condition(m, c, drop_incomplete)
        for k, (ab, ch) in o["lick"].items():
            rows.append({"subject": m, "lick_measure": k, "trials_before": o["nb"], "trials_used": o["nt"],
                         "incomplete": o["n_incomplete"],
                         "lick_abrupt": ab, "da_abrupt": o["da_abrupt"], "lick_change": ch, "da_change": o["da_change"]})
    return pd.DataFrame(rows)


def report_abrupt(tab, brief=False):
    for k, g in tab.groupby("lick_measure", sort=False):
        print(f"\nLick measure: {k}")
        if not brief:
            print(g.drop(columns="lick_measure").round(3).to_string(index=False))
        print(f"abruptness lick {g.lick_abrupt.mean():.3f} against dopamine {g.da_abrupt.mean():.3f}; "
              f"change trial lick {g.lick_change.mean():.1f} against dopamine {g.da_change.mean():.1f}")
        show_t("  paired t, abruptness lick minus dopamine", stats.ttest_rel(g.lick_abrupt, g.da_abrupt))
        show_t("  paired t, change trial lick minus dopamine", stats.ttest_rel(g.lick_change, g.da_change))

## fig4ef-dopamine-leads-behavior

> During initial cue-reward learning, the dopamine response to CS+ develops before anticipatory
> licking, and behavioral learning is much more abrupt than dopaminergic learning (Test 3).

In [ ]:
tab_c1 = abrupt_table(0)
report_abrupt(tab_c1)
print("\nVariant: trials with a NaN dF/F sample from -1 to 2 s left out")
report_abrupt(abrupt_table(0, drop_incomplete=True), brief=True)
print("\nPaper: abruptness t(6) = 9.06, p = 1.0e-4; change trial t(6) = -2.93, p = 0.0263")

**Result: reproduced.** With the Methods' lick measure, lick abruptness exceeds dopamine
abruptness (paired t = 5.06, p = 0.0023) and the dopamine change trial comes earlier (lick minus
dopamine t = 3.31, p = 0.016). With the code's lick measure, t = 9.09 (p = 1.0e-4) and t = 2.93
(p = 0.026), the paper's values up to the sign of the change-trial t, which the paper prints as
dopamine minus lick. Of the trials used, 0 to 3 per animal have a NaN smoothed dF/F sample from
-1 to 2 s (kept with their nansum AUC, as aucsignal.m does); without them the four t values are
5.15, 3.31, 9.15 and 2.93.

## figs10c-cue-response-at-acquisition

> By the session at which behavior was acquired, the dopamine cue response had reached 93% of its
> value on the final session (about 100% relative to the mean of the last two sessions).

In [ ]:
rows = []
for m in jeong.PAV_MICE:
    st, en, ln = jeong.CUMSUM_START[m][0], jeong.CUMSUM_END[m][0], jeong.LEARNED_DAY[m]
    days = [d for d in sorted(pav[m]) if st <= d <= en]
    cr = np.array([np.nanmean(pav[m][d]["early"]) for d in days])
    cs_ = np.array([np.nanmean(pav[m][d]["early"][~pav[m][d]["unsmoothed_incomplete"]]) for d in days])
    x = days.index(ln)
    rows.append({"subject": m, "sessions": len(days), "learned_index": x + 1, "x": cr[x], "n-1": cr[-2], "n": cr[-1],
                 "x_over_n": cr[x] / cr[-1], "x_over_last2": cr[x] / np.mean(cr[-2:]),
                 "incomplete": int(sum(pav[m][d]["unsmoothed_incomplete"][:].sum() for d in (days[x], days[-2], days[-1]))),
                 "strict_x_over_n": cs_[x] / cs_[-1], "strict_x_over_last2": cs_[x] / np.mean(cs_[-2:])})
acq = pd.DataFrame(rows)
print(acq.round(3).to_string(index=False))
print(f"Reading (a), mean of ratios: x/n {acq.x_over_n.mean():.3f}, x/mean(n-1, n) {acq.x_over_last2.mean():.3f}")
print(f"Reading (b), ratio of means: x/n {acq.x.mean() / acq.n.mean():.3f}, x/mean(n-1, n) {acq.x.mean() / ((acq.n + acq['n-1']) / 2).mean():.3f}")
# Variant: the Fig 4 baseline-subtracted 0 to 2 s measure.
var = []
for m in jeong.PAV_MICE:
    st, en, ln = jeong.CUMSUM_START[m][0], jeong.CUMSUM_END[m][0], jeong.LEARNED_DAY[m]
    days = [d for d in sorted(pav[m]) if st <= d <= en]
    cr = np.array([np.nanmean(pav[m][d]["da"]) for d in days]); x = days.index(ln)
    var.append((cr[x] / cr[-1], cr[x] / np.mean(cr[-2:])))
var = np.array(var)
print(f"Variant, Fig 4 measure: mean x/n {var[:, 0].mean():.3f}, x/mean(n-1, n) {var[:, 1].mean():.3f}")
print(f"Without trials with a NaN unsmoothed dF/F sample from -1 to 3 s, reading (a): "
      f"x/n {acq.strict_x_over_n.mean():.3f}, x/mean(n-1, n) {acq.strict_x_over_last2.mean():.3f}")
print("Paper: 0.93 and about 1.0")

**Result: reproduced.** Under reading (a), the code's, the cue response on the learned session is
0.934 of the last session's and 0.934 of the mean of the last two, both within tolerance (0.93 +/-
0.093 and 1.0 +/- 0.1); under reading (b), 0.913 and 0.947. Per animal the ratio to the last
session ranges from 0.59 to 1.33.
One trial of the sessions compared (F2) has a NaN dF/F sample from -1 to 3 s; without it reading
(a) gives 0.938 and 0.938.

## fig4hi-delay-change-behavior-changes

> When the cue is lengthened from 2 s to 8 s (cue-reward delay 3 s to 9 s), animals change the
> timing of anticipatory licking abruptly, and this change is better fit by a changing model than
> by a constant one.

The changing model is fitted by bounded least squares from 20 random start points (Matlab's `fit`
starts custom models at random points; the best fit is kept), with this section's own generator.

In [ ]:
rng_fig4hi = np.random.default_rng(7)
tab_c2 = abrupt_table(1)
report_abrupt(tab_c2)
fits = []
for m in jeong.PAV_MICE:
    o = cumsum_condition(m, 1)
    nb, nt = o["nb"], o["nt"]
    ly, dy = o["lick_series"]["median lick time"][:nt], o["da"][:nt]
    rss_l = jeong.fit_weibull(ly, [0, nb - 10, 0, 0], [9000, nt, 100, 4000], rng_fig4hi)
    rss_d = jeong.fit_weibull(dy, [-2, nb - 10, 0, -10], [50, nt, 10, 10], rng_fig4hi)
    rc_l, rc_d = np.sum((ly - ly.mean()) ** 2), np.sum((dy - dy.mean()) ** 2)
    fits.append({"subject": m, "trials": nt, "dropped_no_lick": o["dropped_no_lick"],
                 "dAIC_lick_k4": jeong.aic(rc_l, 1, nt) - jeong.aic(rss_l, 4, nt),
                 "dAIC_da_k4": jeong.aic(rc_d, 1, nt) - jeong.aic(rss_d, 4, nt)})
    o = cumsum_condition(m, 1, drop_incomplete=True)
    ly, dy = o["lick_series"]["median lick time"][:o["nt"]], o["da"][:o["nt"]]
    fits[-1]["incomplete"] = cumsum_condition(m, 1)["n_incomplete"]
    fits[-1]["dAIC_lick_strict"] = jeong.aic(np.sum((ly - ly.mean()) ** 2), 1, o["nt"]) - jeong.aic(
        jeong.fit_weibull(ly, [0, o["nb"] - 10, 0, 0], [9000, o["nt"], 100, 4000], rng_fig4hi), 4, o["nt"])
    fits[-1]["dAIC_da_strict"] = jeong.aic(np.sum((dy - dy.mean()) ** 2), 1, o["nt"]) - jeong.aic(
        jeong.fit_weibull(dy, [-2, o["nb"] - 10, 0, -10], [50, o["nt"], 10, 10], rng_fig4hi), 4, o["nt"])
fits = pd.DataFrame(fits)
fits["dAIC_lick_k3"] = fits.dAIC_lick_k4 + 2
fits["dAIC_da_k3"] = fits.dAIC_da_k4 + 2
print("\n", fits.round(2).to_string(index=False))
for c in ["dAIC_lick_k4", "dAIC_lick_k3"]:
    print(f"{c}: mean {fits[c].mean():.1f}"); show_t("  one-sample t against 0", stats.ttest_1samp(fits[c], 0))
show_t("dAIC_lick_strict (trials with a NaN dF/F sample from -1 to 2 s left out)", stats.ttest_1samp(fits.dAIC_lick_strict, 0))
print("\nVariant: trials with a NaN dF/F sample from -1 to 2 s left out")
report_abrupt(abrupt_table(1, drop_incomplete=True), brief=True)
print("Paper: abruptness t(6) = 22.92, p = 4.52e-7; lick delta AIC t(6) = 7.49")

**Result: reproduced.** Lick timing changes more abruptly than the dopamine cue response (paired
t = 15.34, p = 4.9e-6; paper t = 22.92), and the changing model fits lick timing better than the
constant one in every animal (mean delta AIC 382.8, t = 7.49, p = 2.9e-4 with k = 4, the paper's
t; t = 7.53 with k = 3).
Of the trials used, 0 to 2 per animal have a NaN smoothed dF/F sample from -1 to 2 s; without
them the abruptness t is 15.40 and the lick delta AIC t is 7.46. The abruptness t is smaller than
the paper's while the delta AIC t equals it; the cause is not found here. One candidate is
HJ-FP-M4, whose Day30, inside the three sessions before the delay change, is not in the dandiset,
so that it has 88 trials before the change where the others have 143 to 150.

## fig4hi-delay-change-dopamine-unchanged

> After the cue-reward delay is lengthened, the dopamine cue response shows no significant change
> (Test 4).

In [ ]:
for c in ["dAIC_da_k4", "dAIC_da_k3"]:
    print(f"{c}: mean {fits[c].mean():.2f}, per animal {fits[c].round(1).tolist()}")
    show_t("  one-sample t against 0", stats.ttest_1samp(fits[c], 0))
print("Paper: t(6) = -0.86, p = 0.42")
# The code's bounds on the dopamine fit hold on the scale assumed here (dF/F as a fraction).
# Variants: no bounds (L and S kept above 0), the bounds applied to dF/F in percent (the stored
# scale, 100 times larger values), and without HJ-FP-M3, whose changing model fits worse than
# the constant one.
rng_fig4hi_da = np.random.default_rng(8)
da_var = []
for m in jeong.PAV_MICE:
    o = cumsum_condition(m, 1)
    nb, nt = o["nb"], o["nt"]
    dy = o["da"][:nt]
    row = {"subject": m, "mean_response": dy.mean(), "sd_response": dy.std()}
    for name, y, bounded in [("unbounded", dy, False), ("percent_scale", dy * 100, True)]:
        rss = jeong.fit_weibull(y, [-2, nb - 10, 0, -10], [50, nt, 10, 10], rng_fig4hi_da, bounded=bounded)
        row[name] = jeong.aic(np.sum((y - y.mean()) ** 2), 1, nt) - jeong.aic(rss, 4, nt)
    da_var.append(row)
da_var = pd.DataFrame(da_var)
print(da_var.round(2).to_string(index=False))
for c in ["unbounded", "percent_scale"]:
    show_t(f"{c}: delta AIC (k = 4) against 0", stats.ttest_1samp(da_var[c], 0))
show_t("fraction scale, bounded, without HJ-FP-M3", stats.ttest_1samp(fits[fits.subject != "HJ-FP-M3"].dAIC_da_k4, 0))
show_t("trials with a NaN dF/F sample from -1 to 2 s left out", stats.ttest_1samp(fits.dAIC_da_strict, 0))

**Result: reproduced, resting on an assumption.** For the dopamine cue response the mean delta
AIC is -18.7 (constant model better on average), t(6) = -0.73, p = 0.49 with k = 4 (paper t =
-0.86, p = 0.42), and t = -0.65, p = 0.54 with k = 3. No change is found; without HJ-FP-M3,
t(5) = 0.99, p = 0.37. With the code's bounds, which the criterion fixes, this test cannot detect
a change in these data. On dF/F as a fraction, the scale used here, b is at most 10 where the
mean response of 5 of the 7 animals is 13 to 45; in three animals delta AIC is exactly -6.0, the
changing model having collapsed to the constant one, and in HJ-FP-M3 the changing model fits
worse than the constant one (-168.5). On dF/F in percent, the stored scale, the bounded model
cannot fit the data at all (delta AIC -272 to -817 in every animal), so that variant does not
test the scale. Fitted without bounds, the changing model fits better in every animal (delta AIC
14 to 38, t = 7.84, p = 2.3e-4). The outcome rests on the assumption that the paper's fit was
bounded as here; the paper's t of -0.86, close to -0.73, suggests it was. The scale the code's
bounds were set for is asked of the authors (finding).

## figs10b-no-omission-response

> After the delay change, there is no suppression of dopamine at the old reward time (3 s) in any
> animal, whereas omission of the reward in extinction does suppress dopamine at the reward time
> (Test 5).

In [ ]:
rows = []
for m in jeong.PAV_MICE:
    d3, dx = jeong.OMISSION_DAYS[m]
    rows.append({"subject": m, "exp3_methods": np.nansum(pav[m][d3]["omit3_base"]), "ext_methods": np.nansum(pav[m][dx]["omit9_base"]),
                 "exp3_code": np.nansum(pav[m][d3]["omit3_code"]), "ext_code": np.nansum(pav[m][dx]["omit9_code"]),
                 "trials": (len(pav[m][d3]["da"]), len(pav[m][dx]["da"])),
                 "incomplete": (int(pav[m][d3]["omit3_incomplete"].sum()), int(pav[m][dx]["omit9_incomplete"].sum())),
                 "exp3_strict": np.nansum(pav[m][d3]["omit3_base"][~pav[m][d3]["omit3_incomplete"]]),
                 "ext_strict": np.nansum(pav[m][dx]["omit9_base"][~pav[m][dx]["omit9_incomplete"]])})
om = pd.DataFrame(rows)
print(om.round(2).to_string(index=False))
for v in ["methods", "code", "strict"]:
    e3, ex = om[f"exp3_{v}"], om[f"ext_{v}"]
    print(f"\n{v}: Exp 3 at or above 0 in {(e3 >= 0).sum()} of 7; extinction below 0 in {(ex < 0).sum()} of 7, mean {ex.mean():.2f}")
    show_t("  paired t, Exp 3 against extinction", stats.ttest_rel(e3, ex))
print("strict: the Methods' measure without trials with a NaN dF/F sample from -1 s to the end of the window")
print("Paper: t = 3.37, p = 0.0150")

**Result: reproduced.** The cumulative response at the old reward time on the first 8 s session
is at or above 0 in all 7 animals, and the extinction omission response is below 0 on average
(Methods' measure, mean -87.25, negative in 6 of 7; code's, -45.75, negative in 5 of 7). The paired
test gives t = 3.03, p = 0.023 (Methods) and t = 3.42, p = 0.014 (code; paper t = 3.37,
p = 0.0150). The extinction response is not negative in every animal, which the criterion's
reading (b) allows.
Leaving out the 0 to 4 trials per session with a NaN dF/F sample in the windows gives t = 3.01,
p = 0.024, with the same signs.

## fig4kl-extinction-abrupt-behavior

> Behavioral learning of extinction is abrupt, much more so than the change of the dopamine cue
> response.

In [ ]:
tab_c3 = abrupt_table(2)
report_abrupt(tab_c3)
print("\nVariant: trials with a NaN dF/F sample from -1 to 2 s left out")
report_abrupt(abrupt_table(2, drop_incomplete=True), brief=True)
print("\nPaper: abruptness t(6) = 5.67, p = 0.0013; change trial t(6) = -2.40, p = 0.0531")

**Result: reproduced.** Lick abruptness exceeds dopamine abruptness with both lick measures
(Methods t = 3.28, p = 0.017; code t = 9.73, p = 6.8e-5; paper t = 5.67). The change trial
comparison, not part of the finding, gives t = -2.95 (p = 0.026) and t = -2.40 (p = 0.0531, the
paper's values).
Without the 0 to 4 trials per animal with a NaN dF/F sample from -1 to 2 s, the abruptness t
values are 3.43 and 9.39. The code variant's abruptness t (9.7) is larger than the paper's 5.67
while its change-trial t equals the paper's; the cause is not found here.

## fig4jl-extinction-cue-response-positive

> The dopamine cue response remains significantly positive well after animals stop licking to the
> cue in extinction (Test 6).

For reading (b) the trials after the lick change trial are those from the change trial (the
change-trial index plus 4, the first point of the distance) to the end of the extinction sessions,
with the code's lick measure, whose change trial matches the paper's test.

In [ ]:
rows = []
for m in jeong.PAV_MICE:
    o = cumsum_condition(m, 2)
    st, en = jeong.CUMSUM_START[m][2], jeong.CUMSUM_END[m][2]
    ext_days = [d for d in o["days"] if st <= d <= en]
    last = pav[m][ext_days[-1]]["da"]
    ch = o["lick"]["code cue+trace"][1] + 4
    ch_m = o["lick"]["Methods 0-2 s"][1] + 4
    after = o["da"][ch:]
    last_inc = pav[m][ext_days[-1]]["da_incomplete"]
    tl = stats.ttest_1samp(last[np.isfinite(last)], 0)
    rows.append({"subject": m, "extinction_days": ext_days, "last_mean": np.nanmean(last), "last_t": tl.statistic,
                 "last_p": tl.pvalue, "last_incomplete": int(last_inc.sum()),
                 "last_mean_strict": np.nanmean(last[~last_inc]), "change_trial": ch, "after_trials": int(np.isfinite(after).sum()),
                 "after_mean": np.nanmean(after), "change_trial_methods": ch_m, "after_mean_methods": np.nanmean(o["da"][ch_m:])})
ej = pd.DataFrame(rows)
print(ej.round(4).to_string(index=False))
show_t("(a) last extinction session, mean over animals against 0", stats.ttest_1samp(ej.last_mean, 0))
show_t("(b) trials after the lick change trial (code's lick measure), mean over animals against 0", stats.ttest_1samp(ej.after_mean, 0))
show_t("(b) trials after the lick change trial (Methods' lick measure), mean over animals against 0", stats.ttest_1samp(ej.after_mean_methods, 0))
show_t("(a) without trials with a NaN dF/F sample from -1 to 2 s", stats.ttest_1samp(ej.last_mean_strict, 0))
print(f"(c) animals positive at p < 0.05 in the last session: {((ej.last_t > 0) & (ej.last_p < 0.05)).sum()} of 7")

**Result: reproduced.** The CS+ response in the last extinction session is positive across animals
(t(6) = 4.31, p = 0.0050; 4.33 without the 2 trials with a NaN dF/F sample), after the lick change
trial (reading b, all 7 animal means positive, t(6) = 4.16, p = 0.006 with the code's lick measure
and t(6) = 3.78, p = 0.0092 with the Methods'), and within each animal's last session (7 of 7 at
p < 0.05).

## fig4p-background-faster-decay

> When unpredicted rewards are added during the ITI, the dopamine cue response stays positive but
> decreases faster over trials than during extinction (Test 7).

In [ ]:
def bgd_vs_ext(start, end, before, P_key="da", strict=False):
    # cumsum_bgdrw_vs_extinction.m for one animal and condition: 140 trials from 50 before the
    # condition starts; readings (a) to (c) and the post-change mean.
    rows = []
    for m in jeong.PAV_MICE:
        row = {"subject": m}
        for c, name in enumerate(["bgd", "ext"]):
            st, en = start[m][c], end[m][c]
            days = [d for d in sorted(pav[m]) if st - before <= d <= en]
            n = sum(len(pav[m][d]["da"]) for d in days[:before])
            sel = np.arange(140) + n - 50
            data = np.concatenate([pav[m][d]["da"] for d in days])[sel]
            inc = np.concatenate([pav[m][d]["da_incomplete"] for d in days])[sel]
            if strict:
                data = np.where(inc, np.nan, data)
            nc = np.nancumsum(data) / np.nansum(data[:50])
            row.update({f"{name}_days": f"{days[0]}-{days[-1]} ({len(days)})", f"{name}_incomplete": int(inc.sum()),
                        f"{name}_pre_sum": np.nansum(data[:50]), f"{name}_a": nc[-1], f"{name}_b": nc[50:].mean(),
                        f"{name}_c": np.nanmean(data[50:]) / np.nanmean(data[:50]), f"{name}_post": np.nanmean(data[50:])})
        rows.append(row)
    return pd.DataFrame(rows)


def report_bgd(bg):
    for r in "abc":
        res = stats.ttest_rel(bg[f"bgd_{r}"], bg[f"ext_{r}"])
        print(f"  reading ({r}): background below extinction in {(bg[f'bgd_{r}'] < bg[f'ext_{r}']).sum()} of 7; "
              f"paired t = {res.statistic:.3f}, p = {res.pvalue:.4g}")


bg = bgd_vs_ext(jeong.BGD_START, jeong.BGD_END, 2)
print(bg.round(3).to_string(index=False))
report_bgd(bg)
show_t("background post-change response against 0", stats.ttest_1samp(bg.bgd_post, 0))
no_m4 = bg[bg.subject != "HJ-FP-M4"]
show_t("variant without HJ-FP-M4 (first background session missing), reading (a)", stats.ttest_rel(no_m4.bgd_a, no_m4.ext_a))
# Readings (a) and (c) are one affine function of the other when no trial is NaN:
# (a) = 1 + (90 / 50) x (c), so they give the same t.
print("largest |(a) - (1 + 1.8 (c))| over animals and conditions:",
      float(np.max(np.abs(np.r_[bg.bgd_a - 1 - 1.8 * bg.bgd_c, bg.ext_a - 1 - 1.8 * bg.ext_c]))))
print("\nVariant: trials with a NaN dF/F sample from -1 to 2 s left out")
report_bgd(bgd_vs_ext(jeong.BGD_START, jeong.BGD_END, 2, strict=True))
# Variant of the sessions: those of cumsum_pavlovian.m, conditions 4 (background) and 3
# (extinction), with one session before each; HJ-FP-M2's background range starts at Day38 there
# against Day33 in cumsum_bgdrw_vs_extinction.m.
start_cp = {m: [jeong.CUMSUM_START[m][3], jeong.CUMSUM_START[m][2]] for m in jeong.PAV_MICE}
end_cp = {m: [jeong.CUMSUM_END[m][3], jeong.CUMSUM_END[m][2]] for m in jeong.PAV_MICE}
bg_cp = bgd_vs_ext(start_cp, end_cp, 1)
print("\nVariant: sessions of cumsum_pavlovian.m, one session before each condition")
print(bg_cp[["subject", "bgd_days", "ext_days", "bgd_pre_sum", "ext_pre_sum", "bgd_a", "ext_a", "bgd_b", "ext_b"]].round(3).to_string(index=False))
report_bgd(bg_cp)
print("Paper: t(6) = -3.51, p = 0.0126")


**Result: not reproduced.** The background reward cue response stays positive after the change
(mean over the 90 trials, t(6) = 3.40, p = 0.015). It decays further than in extinction on
average, but not significantly under any reading: paired t = -2.40, p = 0.053 under readings (a)
and (c), t = -1.97, p = 0.097 under (b) (paper t = -3.51, p = 0.0126); background is below
extinction in 5 of 7 animals. Readings (a) and (c) are one affine function of the other, (a) =
1 + 1.8 (c), so only two independent readings were tested. The criterion asks for background
below extinction at p < 0.05 under every reading, which no reading meets. HJ-FP-M4's first
background reward session (Day38) is not in the dandiset, so its 90 post-change trials start a
session later; without that animal reading (a) gives t(5) = -1.85, p = 0.12. Leaving out the 0
to 4 trials per animal and condition with a NaN dF/F sample, (a) and (c) reach p = 0.048 and 0.047, but (b)
stays at p = 0.088. With the sessions of cumsum_pavlovian.m (conditions 4 and 3, one session
before each), no reading shows a difference (t = 0.69 and 0.72). There HJ-FP-M6's background sum over the 50
trials before the change is 37.3 (288.1 with the sessions of cumsum_bgdrw_vs_extinction.m), so its
reading (a) is 12.15, and HJ-FP-M2's is -492.9, as its range starts at Day38 against Day33.

## fig6b-no-backpropagation

> During initial cue-reward learning, dopamine activity does not shift backward from just before
> reward to the cue; the late (pre-reward) response does not differ from the early (cue) response
> in the first 100 trials (Test 9).

In [ ]:
rows = []
for m in jeong.PAV_MICE:
    st, en = jeong.CUMSUM_START[m][0], jeong.CUMSUM_END[m][0]
    days = [d for d in sorted(pav[m]) if st <= d <= en]
    e = np.concatenate([pav[m][d]["early"] for d in days])
    l = np.concatenate([pav[m][d]["late"] for d in days])
    p = np.concatenate([pav[m][d]["pre"] for d in days])
    inc = np.concatenate([pav[m][d]["unsmoothed_incomplete"] for d in days])
    eb, lb = e - p, l - p
    s_ = lambda v: np.where(inc, np.nan, v)
    rows.append({"subject": m, "trials": len(e), "incomplete": int(inc[:100].sum() + inc[350:400].sum()),
                 "methods": np.nanmean(lb[:100] - eb[:100]) / np.nanmean(eb[350:400]),
                 "code": np.nanmean(l[:100] - e[:100]) / np.nanmean(e[350:400]),
                 "methods_strict": np.nanmean(s_(lb)[:100] - s_(eb)[:100]) / np.nanmean(s_(eb)[350:400])})
bp = pd.DataFrame(rows)
print(bp.round(3).to_string(index=False))
for v in ["methods", "code", "methods_strict"]:
    print(f"{v}: mean {bp[v].mean():.3f}"); show_t("  one-sample t of late minus early against 0", stats.ttest_1samp(bp[v], 0))
print("methods_strict leaves out trials with a NaN unsmoothed dF/F sample from -1 to 3 s")
print("Paper: t = -0.16, p = 0.8771")

**Result: reproduced.** Over the first 100 CS+ trials the normalized late minus early response is
-0.008 with the Methods' baseline (t = -0.23, p = 0.83) and -0.006 without it (t = -0.16,
p = 0.876, the paper's values). No difference is found.
Leaving out the 0 to 4 trials per animal with a NaN dF/F sample from -1 to 3 s gives t = -0.22,
p = 0.84.

## Experiment 6: trial-less conditioning

Sessions from learnedday to endday of intermediate_rsp_trialless.m; `jeong.trialless_pairs`
follows that script and the Methods.

In [ ]:
peak_readings = {(0.5, "last"): 1, (0.5, "mean"): 1, (1.0, "last"): 1, (1.0, "mean"): 1}
rows = []
for m in jeong.PAV_MICE:
    for s in subject_sessions(m, "Poisson"):
        if jeong.TRIALLESS_LEARNED[m] <= s["day"] <= jeong.TRIALLESS_END[m]:
            rows.append({"subject": m, "day": s["day"], **jeong.trialless_pairs(s, peak_readings)})
tl = pd.DataFrame(rows)
print(tl[["subject", "day", "n_cues", "n_pairs", "delay", "cue_prev", "cue_inter", "rew_prev", "rew_inter"]].round(3).to_string(index=False))
tla = tl.groupby("subject").mean(numeric_only=True)
print(f"Cues of pairs with a NaN dF/F sample from -0.5 to 0.5 s: {tl.n_incomplete_cue.sum()} of {2 * tl.n_pairs.sum()}; "
      f"their rewards with one from 0 to 0.5 s after the first lick: {tl.n_incomplete_rew.sum()}")
# Variant: those cues and rewards left out of the means.
tl_strict = pd.DataFrame([{"subject": m, **jeong.trialless_pairs(s, peak_readings, drop_incomplete=True)}
                          for m in jeong.PAV_MICE for s in subject_sessions(m, "Poisson")
                          if jeong.TRIALLESS_LEARNED[m] <= s["day"] <= jeong.TRIALLESS_END[m]])
tla_strict = tl_strict.groupby("subject").mean(numeric_only=True)

## fig5d-intermediate-cue-positive

> In the trial-less task, the dopamine response to an intermediate cue (one arriving before the
> previous cue's reward) is positive but weaker than the response to the previous cue (Test 8).

In [ ]:
ratio_cue = tla.cue_inter / tla.cue_prev
print("Per animal ratio intermediate over previous cue response:", ratio_cue.round(3).to_dict())
print(f"mean ratio {ratio_cue.mean():.3f}; pairs per animal {tl.groupby('subject').n_pairs.sum().to_dict()}")
show_t("one-sample t against 0", stats.ttest_1samp(ratio_cue, 0))
show_t("one-sample t against 1", stats.ttest_1samp(ratio_cue, 1))
print("Paper: t(6) = 6.64, p = 5.6e-4")
r_strict = tla_strict.cue_inter / tla_strict.cue_prev
print(f"Without cues with a NaN dF/F sample in their windows: mean ratio {r_strict.mean():.3f}")
show_t("  one-sample t against 0", stats.ttest_1samp(r_strict, 0))

**Result: reproduced.** The intermediate cue response is positive relative to the previous cue's
baseline, at 0.435 of the previous cue's response (0.12 to 0.68 per animal), t(6) = 6.66,
p = 5.6e-4 against 0 (paper t = 6.64), and below 1.
No cue or reward of the pairs has a NaN dF/F sample in its windows (0 of 190 cues), so leaving
such values out changes nothing.

## fig5d-intermediate-reward-larger

> The dopamine response to the reward paired with an intermediate cue is larger than the response
> to the reward of the previous cue (the ratio is above 1).

In [ ]:
ratio_rw = tla.rew_inter / tla.rew_prev
print("Per animal ratio intermediate over previous reward response:", ratio_rw.round(3).to_dict())
print(f"mean ratio {ratio_rw.mean():.3f}")
show_t("one-sample t against 1", stats.ttest_1samp(ratio_rw, 1))
print("Paper: t(6) = 2.95, p = 0.0256")
r_strict = tla_strict.rew_inter / tla_strict.rew_prev
print(f"Without rewards with a NaN dF/F sample in their windows: mean ratio {r_strict.mean():.3f}")
show_t("  one-sample t against 1", stats.ttest_1samp(r_strict, 1))

**Result: reproduced.** The mean ratio is 1.85 (1.08 to 3.25), t(6) = 2.96, p = 0.025 against 1
(paper t = 2.95, p = 0.0256).
No cue or reward of the pairs has a NaN dF/F sample in its windows (0 of 190 cues), so leaving
such values out changes nothing.

## figs11de-intermediate-cue-local-baseline

> With the baseline taken immediately before the intermediate cue and the response measured as a
> peak, the intermediate cue response is still positive.

In [ ]:
for pw, bw in peak_readings:
    r = tla[f"peak_{pw}_{bw}_inter"] / tla[f"peak_{pw}_{bw}_prev"]
    print(f"peak over 0 to {pw} s, baseline {'last sample' if bw == 'last' else 'mean of 0.5 s'} before the cue: "
          f"mean ratio {r.mean():.3f}, min {r.min():.3f}")
    show_t("  one-sample t against 0", stats.ttest_1samp(r, 0))
print("Paper: t = 10.42, p = 4.6e-5")
print("Without cues with a NaN dF/F sample in their windows:")
for pw, bw in peak_readings:
    r = tla_strict[f"peak_{pw}_{bw}_inter"] / tla_strict[f"peak_{pw}_{bw}_prev"]
    res = stats.ttest_1samp(r, 0)
    print(f"  peak 0 to {pw} s, baseline {bw}: mean ratio {r.mean():.3f}, t = {res.statistic:.3f}, p = {res.pvalue:.3g}")

**Result: reproduced.** Under all four readings the intermediate cue's peak response over the
previous cue's is positive (mean ratio 0.25 to 0.45, t = 5.21 to 8.35, p <= 0.002; paper t =
10.42).
No cue or reward of the pairs has a NaN dF/F sample in its windows (0 of 190 cues), so leaving
such values out changes nothing.

## Experiment 7: sequential conditioning with inhibition

Per animal the first session in Day order is the random reward session and the rest are
conditioning sessions (checked in methods-session-counts). DAT-Cre are datHT-M2 to M7 (M8
excluded, Fig S12D) and controls datWT-F1 and WT-F1, F2, F3, M1, M2, M3.

In [ ]:
e7 = {}
for m in jeong.DAT_MICE + [jeong.DAT_EXCLUDED] + jeong.CONTROL_MICE:
    ss = subject_sessions(m)
    e7[m] = {"random": jeong.random_session_reward(ss[0]), "cond": [jeong.seq_session(s) for s in ss[1:]]}
    cues = [(len(c["cs1"]), len(c["cs2"])) for c in e7[m]["cond"]]
    print(m, "conditioning sessions", len(cues), "CS1/CS2 per session", cues)
# auc_python gives NaN when a window holds no dF/F sample; the session means below use nanmean,
# so such trials drop out. Count them per measure, for sessions where any are NaN.
nan_rows = []
for m in e7:
    for i, c in enumerate(e7[m]["cond"]):
        row = {"subject": m, "conditioning_session": i + 1, "of": len(e7[m]["cond"]), "cs1": len(c["cs1"])}
        row.update({k: int(np.isnan(c[k]).sum()) for k in ["cs1_15", "early", "late", "rw_15"]})
        if any(row[k] for k in ["cs1_15", "early", "late", "rw_15"]):
            nan_rows.append(row)
print("\nConditioning sessions with trials without a dF/F value (NaN AUC):")
print(pd.DataFrame(nan_rows).to_string(index=False))
print("HJ-FP-WT-stGtACR-M2, last session, CS1 trials (1-based) without a value:",
      (np.flatnonzero(np.isnan(e7["HJ-FP-WT-stGtACR-M2"]["cond"][-1]["cs1_15"])) + 1).tolist())
print("Random reward sessions with a NaN reward response:", [m for m in e7 if not np.isfinite(e7[m]["random"]["rw_15"])])

In Experiment 7 a trial whose window holds no dF/F sample has no value (`auc_python`, as
calculate_auc), and the session means leave it out. HJ-FP-WT-stGtACR-M2's last conditioning
session (Day6) has no dF/F for its first 1117 s, so its CS1 and CS2 trials 1 to 17 and 18 of its
reward responses have no value. Elsewhere only reward responses are missing: 1, 12 and 2 in one
session each of datHT-M3, datHT-M5 and WT-M1 (in WT-M1's first session, so its day 1 reward
response, which normalises Fig 6I, is a mean over 48 trials). Of the trials Fig 6D uses (1 to 50 and 151 to
200), only trial 200 of WT-M2 falls in that session (its sessions hold 50, 49, 50, 50 and 50
trials), and it has no value. fig6i-learning-despite-inhibition
and fig6i-lower-cs1-asymptote take that session's mean over its other 33 trials, and the latter
shows the result without the session.

## fig6d-cues-increase-together

> In sequential conditioning without inhibition, the dopamine responses to CS2 and CS1 increase
> together early in learning (no difference in the first 50 trials) (Test 10).

In [ ]:
rows = []
for m in jeong.CONTROL_MICE:
    c = e7[m]["cond"]
    e = np.concatenate([x["early"] for x in c]); l = np.concatenate([x["late"] for x in c])
    p1 = np.concatenate([x["pre1"] for x in c]); p15 = np.concatenate([x["pre15"] for x in c])
    row = {"subject": m, "trials": len(e)}
    for name, b in [("code (no baseline)", 0 * e), ("Methods, 1 s baseline", p1), ("Methods, 1.5 s baseline", p15)]:
        E, L = e - b, l - b
        row[name] = np.nanmean(L[:50] - E[:50]) / np.nanmean(E[150:200])
        row[name + " late ratio"] = np.nanmean(L[150:200]) / np.nanmean(E[150:200])
    rows.append(row)
sq = pd.DataFrame(rows)
print(sq.round(3).to_string(index=False))
for name in ["code (no baseline)", "Methods, 1 s baseline", "Methods, 1.5 s baseline"]:
    print(f"{name}: mean CS2 minus CS1 over the first 50 trials {sq[name].mean():.3f}")
    show_t("  one-sample (paired) t against 0", stats.ttest_1samp(sq[name], 0))
print("Paper: t = -0.03, p = 0.9752")

**Result: reproduced.** Over the first 50 trials the normalized CS2 minus CS1 response does not
differ from 0 under any reading: t = -0.03, p = 0.975 without a baseline (the paper's values), and
p = 0.98 and 0.99 with the 1 s and 1.5 s baselines (means -0.003 to -0.001).

## fig6d-cs2-below-cs1-late

> Later in sequential conditioning, the CS2 response falls below the CS1 response (the two
> separate).

In [ ]:
for name in ["code (no baseline)", "Methods, 1 s baseline", "Methods, 1.5 s baseline"]:
    v = sq[name + " late ratio"]
    print(f"{name}: CS2/CS1 over trials 151 to 200, mean {v.mean():.3f} (range {v.min():.3f} to {v.max():.3f})")
    show_t("  one-sample t against 1", stats.ttest_1samp(v, 1))

**Result: reproduced.** Over trials 151 to 200 the CS2 response is 0.55 of the CS1 response
(0.30 to 0.71 per animal), t(6) = -8.01, p = 2.0e-4 against 1, and below 1 under the baseline
readings too.

## fig6g-inhibition-magnitude

> Optogenetic inhibition of VTA dopamine neurons strongly suppresses NAcc dopamine during the
> CS2-to-reward delay in DAT-Cre mice (to about -0.6 times the reward response on day 1) and
> reduces their reward response, compared with controls.

The baseline is the mean dF/F over the 0.5 s before CS1 (claim), or before every code-15 event,
CS1 and CS2 (the code's `reftimes`, a variant). For a window of another length the baseline AUC is
scaled to that length, so that each response subtracts a baseline over an equal window (claims
header); this is a choice made here, and it only matters for the 1 s reward window.

In [ ]:
rows = []
for m in jeong.DAT_MICE + jeong.CONTROL_MICE + [jeong.DAT_EXCLUDED]:
    r, c = e7[m]["random"], e7[m]["cond"][0]
    row = {"subject": m, "group": "DAT-Cre" if "datHT" in m else "control"}
    for bname, b in [("cs1", c["base05_cs1"]), ("all15", c["base05_all15"])]:
        inh = c["inh_bins"][1.0] - b
        rw1, rw05 = c["rw1"] - 2 * b, c["rw05"] - b
        row[f"{bname}: (a) AUC ratio, 1 s reward"] = inh / rw1
        row[f"{bname}: (b) AUC ratio, 0.5 s reward"] = inh / rw05
        row[f"{bname}: (c) level ratio, 1 s reward"] = (inh / 0.5) / rw1
        row[f"{bname}: day 1 reward, 1 s"] = rw1 / (r["rw1"] - 2 * r["base05"])
        row[f"{bname}: day 1 reward, 0.5 s"] = rw05 / (r["rw05"] - r["base05"])
    rows.append(row)
inh = pd.DataFrame(rows)
print(inh.round(3).T.to_string())
main = inh[inh.subject != jeong.DAT_EXCLUDED]
for col in [c for c in inh.columns if ":" in c]:
    d, w = main[main.group == "DAT-Cre"][col], main[main.group == "control"][col]
    res = stats.ttest_ind(d, w)
    print(f"{col:42s} DAT-Cre mean {d.mean():7.3f}  control {w.mean():7.3f}  t = {res.statistic:6.2f}, p = {res.pvalue:.3g}")
print("Paper: inhibition t = -5.74, p = 1.3e-4; day 1 reward t = -3.99, p = 0.0021; DAT-Cre about -0.6")

**Result: reproduced.** With the baseline before CS1, dopamine 1 to 1.5 s after CS2 is lower in
the 6 DAT-Cre mice than in the 7 controls (t = -8.62, p = 3.2e-6 under readings a and c; t =
-6.86, p = 2.7e-5 under b), and the day 1 reward response relative to the random reward session is
lower (1 s window t = -4.13, p = 0.0017). The DAT-Cre mean is -0.565 under reading (b), within
tolerance of -0.6, and -0.247 and -0.494 under (a) and (c). With the code's baseline over all
code-15 events the tests give t = -5.74 (p = 1.3e-4) and, with the 0.5 s reward window, t = -3.99
(p = 0.0021), the paper's values, and the DAT-Cre mean is -0.632 under (b).

## fig6i-learning-despite-inhibition

> Despite inhibition of dopamine from CS2 to reward, every DAT-Cre animal learns anticipatory
> licking and acquires a positive dopamine response to CS1, both increasing over sessions as in
> controls (Test 11).

In [ ]:
lc_rows, reg = [], []
for grp, mice in [("control", jeong.CONTROL_MICE), ("DAT-Cre", jeong.DAT_MICE)]:
    X, Y, L = [], [], []
    for m in mice:
        c = e7[m]["cond"]
        cs = [np.nanmean(x["cs1_15"]) for x in c]
        rw = np.nanmean(c[0]["rw_15"])
        lk = [np.nanmean(x["lick_cs1_rw"]) for x in c]
        v = np.array(cs[:3] + cs[-2:]) / rw
        l = np.array(lk[:3] + lk[-2:])
        X += list(range(5)); Y += list(v); L += list(l)
        lc_rows.append({"group": grp, "subject": m, "sessions": len(c), **{f"cs1_{i}": v[i] for i in range(5)},
                        **{f"lick_{i}": l[i] for i in range(5)},
                        "s12_first": np.nanmean(c[0]["lick_cs1_cs2"]), "s12_last": np.nanmean(c[-1]["lick_cs1_cs2"]),
                        "s12_code_first": np.nanmean(c[0]["lick_cs1_cs2_code"]), "s12_code_last": np.nanmean(c[-1]["lick_cs1_cs2_code"])})
    for name, y in [("CS1 response", Y), ("anticipatory licks", L)]:
        slope, t, p, n = ols_slope(X, y)
        reg.append({"group": grp, "measure": name, "slope": slope, "t": t, "p": p, "n": n})
lc = pd.DataFrame(lc_rows)
print(lc.round(3).to_string(index=False))
print(pd.DataFrame(reg).round(6).to_string(index=False))
dat = lc[lc.group == "DAT-Cre"]
print(f"DAT-Cre animals with last-session CS1 response > 0: {(dat.cs1_4 > 0).sum()} of {len(dat)}; licks > 0: {(dat.lick_4 > 0).sum()} of {len(dat)}")
print("Paper: CS1 t = 5.49 (WT), 2.55 (DAT-Cre); licks t = 3.61 (WT), 4.92 (DAT-Cre)")

**Result: reproduced.** Over the first three and last two sessions, the normalized CS1 response
rises in controls (t = 4.17, p = 2.1e-4, 35 values) and in DAT-Cre mice (t = 2.87, p = 0.0077, 30
values), and so do anticipatory licks (t = 3.67, p = 8.4e-4; t = 4.22, p = 2.3e-4). In the last
session every DAT-Cre animal has a positive CS1 response and positive anticipatory licking.

## fig6i-lower-cs1-asymptote

> With inhibition, the CS1 dopamine response reaches a lower level in the last session than in
> controls.

In [ ]:
w, d = lc[lc.group == "control"].cs1_4, lc[lc.group == "DAT-Cre"].cs1_4
print(f"last-session normalized CS1 response: control mean {w.mean():.3f} (n {len(w)}), DAT-Cre {d.mean():.3f} (n {len(d)})")
show_t("two-sample t, control against DAT-Cre", stats.ttest_ind(w, d))
print("Paper: t = 2.80, p = 0.0171")

# Variants of the session and normalisation choices: the last session's CS1 response normalised
# by the random reward session's reward response (in place of the first conditioning session's),
# not normalised, with HJ-FP-WT-stGtACR-M3's last session (the shortest) replaced by the one
# before it, and with HJ-FP-WT-stGtACR-M2's last session (Day6, no dF/F for its first 1117 s)
# replaced by the one before it.
var = []
for grp, mice in [("control", jeong.CONTROL_MICE), ("DAT-Cre", jeong.DAT_MICE)]:
    for m in mice:
        c = e7[m]["cond"]
        cs = [np.nanmean(x["cs1_15"]) for x in c]
        last = -2 if m == "HJ-FP-WT-stGtACR-M3" else -1
        last_m2 = -2 if m == "HJ-FP-WT-stGtACR-M2" else -1
        var.append({"group": grp, "subject": m, "last_trials": len(c[-1]["cs1"]),
                    "last_nan": int(np.isnan(c[-1]["cs1_15"]).sum()),
                    "first_rw": np.nanmean(c[0]["rw_15"]), "random_rw": e7[m]["random"]["rw_15"],
                    "raw_last": cs[-1], "claim": cs[-1] / np.nanmean(c[0]["rw_15"]),
                    "by_random_rw": cs[-1] / e7[m]["random"]["rw_15"],
                    "m3_previous": cs[last] / np.nanmean(c[0]["rw_15"]),
                    "m2_previous": cs[last_m2] / np.nanmean(c[0]["rw_15"])})
var = pd.DataFrame(var)
print(var.round(3).to_string(index=False))
for col in ["claim", "by_random_rw", "raw_last", "m3_previous", "m2_previous"]:
    w, d = var[var.group == "control"][col], var[var.group == "DAT-Cre"][col]
    print(f"{col}: control mean {w.mean():.3f}, DAT-Cre {d.mean():.3f}"); show_t("  two-sample t", stats.ttest_ind(w, d))

**Result: not reproduced.** The last-session CS1 response is lower in DAT-Cre mice on average
(0.359 against 0.690 of the first conditioning session's reward response) but not significantly,
t(11) = 1.70, p = 0.117 (paper t = 2.80, p = 0.0171). One control (WT-M3, 0.077) is below every
DAT-Cre value; its last session has 25 trials, where every other animal's has 50. The outcome
turns on these choices: with WT-M3's previous session in place of its last, t = 2.43 (p = 0.033);
normalised by the random reward session's reward response, t = 2.84 (p = 0.016); not normalised,
t = 2.85 (p = 0.016). WT-M2's last session has no dF/F for its first 1117 s, so 17 of its 50 CS1
trials have no value and are left out of its mean; with its previous session in place of it, its
value is 0.462 against 0.459 and t = 1.70 (p = 0.117), so these trials do not bear on the outcome. The normalisation of the criterion is the one Supplementary Note 10 and
learnign_curve_sequential.py name ("normalized by reward response on day 1 of conditioning"), and
no passage excludes a short session, so the outcome is kept on the criterion's measure.

## fig6i-last-session-licking-similar

> In the last session, anticipatory licking does not differ between DAT-Cre and control animals.

In [ ]:
w, d = lc[lc.group == "control"].lick_4, lc[lc.group == "DAT-Cre"].lick_4
print(f"last-session licks (CS1 to reward minus baseline): control mean {w.mean():.2f}, DAT-Cre {d.mean():.2f}, difference {w.mean() - d.mean():.2f}")
show_t("two-sample t, control against DAT-Cre", stats.ttest_ind(w, d))
print("Paper: t = 0.88, p = 0.3999")

**Result: reproduced.** Last-session anticipatory licking is 7.25 licks in controls and 4.84 in
DAT-Cre mice, t(11) = 1.17, p = 0.27 (paper t = 0.88, p = 0.40): no difference is found.

## figs12f-pre-cs2-licking

> Anticipatory licking between CS1 and CS2, before inhibition starts, increases from the first to
> the last session (main effect of session), with no effect of genotype and no genotype by session
> interaction.

In [ ]:
for label, a, b in [("claim (baseline 1.5 s before CS1)", "s12_first", "s12_last"),
                    ("code variant (baseline 1500 s before CS1)", "s12_code_first", "s12_code_last")]:
    long = pd.concat([lc[["subject", "group", a]].rename(columns={a: "licks"}).assign(session="first"),
                      lc[["subject", "group", b]].rename(columns={b: "licks"}).assign(session="last")])
    print(f"\n{label}: mean first {lc[a].mean():.2f}, last {lc[b].mean():.2f}")
    print(pg.mixed_anova(long, dv="licks", within="session", between="group", subject="subject").round(4).to_string(index=False))
print("\nPaper: session F = 9.07, p = 0.0118; genotype F = 0.25, p = 0.6241; interaction F = 0.07, p = 0.7915")

**Result: reproduced.** Pre-CS2 licking rises from the first to the last session (from 0.33 to
1.62 licks; session F(1, 11) = 7.39, p = 0.020), with no genotype effect (F = 0.57, p = 0.47) and
no interaction (F = 0.0003, p = 0.99). With the code's baseline window the session effect is
F = 5.23, p = 0.043, and the others stay non-significant.

## Simulation claims (blocked)

fig2-anccr-classic-results, fig3cf-simulated-unpredicted-rewards, fig5b-simulated-intermediate-cue,
fig6h-simulated-inhibition, figs6-learns-causal-structure, figs5-timescale-invariance and
figs13-ramping are simulations of the ANCCR and TDRL models. They use no recorded data, and the
paper's simulation code is Matlab (simulation/ and functions/calculateANCCR.m of the ANCCR
repository). Matlab cannot be installed in the entry's locked Python environment, and GNU Octave,
which might run the scripts, is not on the runner and cannot be installed from the lock file
either. A port of the models to Python was not attempted. They are blocked as not_runnable, the
protocol's reason for a method whose software cannot be run here; nothing is computed for them.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Data | Outcome |
|---|---|---|
| session-day-link | with labels | partially reproduced |
| methods-session-counts | with labels | reproduced |
| exp1-excluded-rewards | dataset | reproduced |
| fig3e-reward-response-increases | dataset | reproduced |
| fig3g-response-iri-positive | dataset | reproduced |
| figs8k-iri-correlation-stable | dataset | reproduced |
| figs8b-lick-rates-constant | dataset | reproduced |
| figs8d-consummatory-lick-uncorrelated | dataset | reproduced |
| figs8e-lick-bout-responses | dataset | blocked: methods underspecified |
| figs8g-baseline-unchanged | dataset | reproduced |
| figs8h-baseline-uncorrelated-with-reward-change | dataset | reproduced |
| figs8l-lick-rate-tracks-iri | dataset | reproduced |
| fig4ef-dopamine-leads-behavior | with labels | reproduced |
| figs10c-cue-response-at-acquisition | with labels | reproduced |
| fig4hi-delay-change-behavior-changes | with labels | reproduced |
| fig4hi-delay-change-dopamine-unchanged | with labels | reproduced |
| figs10b-no-omission-response | with labels | reproduced |
| fig4kl-extinction-abrupt-behavior | with labels | reproduced |
| fig4jl-extinction-cue-response-positive | with labels | reproduced |
| fig4p-background-faster-decay | with labels | not reproduced |
| fig5d-intermediate-cue-positive | with labels | reproduced |
| fig5d-intermediate-reward-larger | with labels | reproduced |
| figs11de-intermediate-cue-local-baseline | with labels | reproduced |
| fig6b-no-backpropagation | with labels | reproduced |
| fig6d-cues-increase-together | dataset | reproduced |
| fig6d-cs2-below-cs1-late | dataset | reproduced |
| fig6g-inhibition-magnitude | dataset | reproduced |
| fig6i-learning-despite-inhibition | dataset | reproduced |
| fig6i-lower-cs1-asymptote | dataset | not reproduced |
| fig6i-last-session-licking-similar | dataset | reproduced |
| figs12f-pre-cs2-licking | dataset | reproduced |
| seven simulation claims | dataset | blocked: not runnable |

**How the analysis differs from the paper's.** The measures follow the paper's code at the commit
the claims file cites, ported to Python; where the Methods and the code differ, both are shown.
The Matlab curve fits start at random points; here the best of 20 seeded starts is kept. dF/F is
divided by 100 for the Matlab-derived measures, as the files store percent and the Methods
define dF/F as a fraction. The code's bounds on the dopamine model bind on either scale, so the
dopamine model comparison (fig4hi-delay-change-dopamine-unchanged) cannot detect a change here;
without the bounds it finds one in every animal. That outcome rests on the assumption that the
paper's fit was bounded as here. As in the code, trials with NaN dF/F samples
in their windows keep the nansum of the others (0 outside the coverage); they are 37 of 8721 CS+
trials, and leaving them out changes no outcome. In Experiment 7, where a trial with no dF/F
sample in its window has no value and drops out of the session mean, 17 of the 50 CS1 trials of
HJ-FP-WT-stGtACR-M2's last session are such trials; replacing that session changes no outcome. Seven integer days inside the code's day ranges have no
session in the dandiset, and the analysis runs on the sessions present.

**What was found about the dandiset.**

- Event times are in seconds, though the files state milliseconds.
- Seven integer days within the code's day ranges have no session in the dandiset (M2 Day23, M3
  Day16 and Day31, M4 Day30 and Day38, F2 Day16, M6 Day49). M4 Day38 is a listed start day, its
  first background reward session, so that session is missing; the paper's session counts of
  Experiments 2, 4 and 5 equal the counts with all seven days, which suggests the others are
  missing sessions too.
- In the large Experiment 7 files the raw photometry timestamps are in thousands of seconds,
  though stated in seconds.
- The DAT-Cre mouse HJ-FP-datHT-stGtACR-M7 has genotype `wt` in its file.
- Some sessions lose the start or the end of their photometry relative to the event log.
- No recording is deposited twice; dF/F and the event log are consistent with one clock, except
  in 27 sessions with no clear response to the reference event, where the check cannot show it.

In [ ]:
print(f"Notebook run time: {(time.time() - started) / 60:.1f} min")